# 02 - Data Cleaning

## Insurance Analytics Platform

This notebook focuses on cleaning and standardizing the raw insurance datasets after the initial data-understanding stage.

The main objectives are to:

- Preserve the original raw datasets without modification.
- Standardize monetary variables and convert them to numeric data types.
- Parse and validate mixed-format date variables.
- Standardize vehicle power measurements.
- Resolve inconsistent categorical representations.
- Handle missing values according to their business meaning.
- Separate composite fields where appropriate.
- Apply business-rule validation checks.
- Produce clean datasets suitable for MySQL, Power BI, and machine learning.

Cleaning decisions are based on the findings documented in `01_data_understanding.ipynb`.

Whenever possible, reusable transformations will later be moved into the `src/preprocessing/` module.

In [19]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Project Paths and Raw Data Loading

Project-relative paths are defined using `pathlib` so that the notebook remains portable across different operating systems and environments.

The raw datasets are loaded from `data/raw/`.

The original DataFrames are preserved, while separate working copies are created for all cleaning operations.

In [3]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Project root       : {PROJECT_ROOT}")
print(f"Raw data directory : {RAW_DATA_DIR}")
print(f"Processed directory: {PROCESSED_DATA_DIR}")

Project root       : c:\Users\okand\Desktop\Projects\insurance-analytics-platform
Raw data directory : c:\Users\okand\Desktop\Projects\insurance-analytics-platform\data\raw
Processed directory: c:\Users\okand\Desktop\Projects\insurance-analytics-platform\data\processed


In [4]:
CONTRACTS_PATH = RAW_DATA_DIR / "contracts.csv"
CLAIMS_PATH = RAW_DATA_DIR / "claims.csv"
VEHICLES_PATH = RAW_DATA_DIR / "vehicles.csv"

source_files = {
    "contracts": CONTRACTS_PATH,
    "claims": CLAIMS_PATH,
    "vehicles": VEHICLES_PATH,
}

for name, path in source_files.items():
    print(f"{name:<10} -> {path.exists()} | {path.name}")

contracts  -> True | contracts.csv
claims     -> True | claims.csv
vehicles   -> True | vehicles.csv


In [5]:
contracts_raw = pd.read_csv(CONTRACTS_PATH)
claims_raw = pd.read_csv(CLAIMS_PATH)
vehicles_raw = pd.read_csv(VEHICLES_PATH)

### Working Copies

Separate copies of the raw datasets are created for cleaning.

The `_raw` DataFrames will remain unchanged and can be used throughout the notebook to validate that transformations have not altered the original source data.

In [6]:
contracts = contracts_raw.copy()
claims = claims_raw.copy()
vehicles = vehicles_raw.copy()

In [7]:
datasets = {
    "contracts": contracts,
    "claims": claims,
    "vehicles": vehicles
}

for name, df in datasets.items():
    print(
        f"{name:<10}: "
        f"{df.shape[0]:,} rows x {df.shape[1]} columns"
    )

contracts : 15,000 rows x 14 columns
claims    : 155 rows x 10 columns
vehicles  : 5,390 rows x 10 columns


In [8]:
print(
    "Contracts identical:",
    contracts.equals(contracts_raw)
)

print(
    "Claims identical:",
    claims.equals(claims_raw)
)

print(
    "Vehicles identical:",
    vehicles.equals(vehicles_raw)
)

Contracts identical: True
Claims identical: True
Vehicles identical: True


## 2. Monetary Variable Standardization

Several monetary variables are stored as strings because the raw data contains inconsistent currency representations.

Examples identified during the data-understanding stage include:

- `1250.50€`
- `€1250.50`
- `$1250.50`
- `1250.50 EUR`
- `1250.50 euros`
- `1250`

Previous analysis indicated that these representations behave as formatting inconsistencies rather than distinct currencies.

The cleaning strategy therefore:

- Preserves missing values.
- Removes currency symbols and textual currency indicators.
- Preserves the underlying numeric amount.
- Converts the cleaned values to numeric data types.
- Validates that no additional missing values are introduced during parsing.

No currency conversion is applied.

In [10]:
def parse_monetary_series(series: pd.Series) -> pd.Series:
    """
    Convert monetary strings into numeric values while preserving
    original missing values.
    """

    cleaned = (
        series
        .astype("string")
        .str.strip()
        .str.replace(r"[^\d.\-]", "", regex=True)
    )

    cleaned = cleaned.replace("", pd.NA)

    return pd.to_numeric(
        cleaned,
        errors="coerce"
    )

In [11]:
test_values = pd.Series(
    [
        "1974.98€",
        "€620.93",
        "$1568.11",
        "878.30 EUR",
        "1048.35 euros",
        "967",
        None,
    ]
)

pd.DataFrame(
    {
        "raw_value": test_values,
        "parsed_value": parse_monetary_series(test_values),
    }
)

,raw_value,parsed_value
0,1974.98€,"1,974.98"
1,€620.93,620.93
2,$1568.11,"1,568.11"
3,878.30 EUR,878.30
4,1048.35 euros,"1,048.35"
5,967,967.00
6,NaN,<NA>


In [12]:
contracts.head()

,contract_id,client_id,client_name,product,start_date,end_date,annual_premium,status,city_postal,risk_zone,client_age,channel,csp,gender
0,CTR_000001,CLI_000001,Pascal Dubois,Life,11/08/2023,2024-09-08,1974.98€,Renewed,Paris_75001,High,50.00,Agency,NaN,F
1,CTR_000002,CLI_000002,Sophie Simon,Auto,2025-08-12,2026-08-15,€620.93,Active,Bordeaux_33000,Medium,43.00,Phone,Worker,F
2,CTR_000003,CLI_000003,Olivier Durand,Auto,2025-06-14,2026-06-30,$1568.11,Suspended,Paris_75001,High,63.00,Broker,NaN,Male
3,CTR_000004,CLI_000004,Sandrine Michel,Life,04/17/2023,2024-04-20,€1752.17,Renewed,Bordeaux_33000,Medium,54.00,Broker,Manager,Male
4,CTR_000005,CLI_000005,Pierre Durand,Auto,2025-03-02,2026-02-26,$977.59,Suspended,Marseille_13000,Medium,39.00,Web,Employee,M


In [13]:
claims.head()

,claim_id,contract_id,occurrence_date,declaration_date,claim_type,damage_amount,indemnified_amount,status,expert_id,liability
0,CLM_0000001,CTR_008899,26-11-2023,2023-10-02,Theft,15213.03€,10977.27€,Closed,EXP_013,Third_party
1,CLM_0000002,CTR_001770,2025-08-26,2025-08-28,Fire,2321.55€,NaN,Expert_review,EXP_013,Third_party
2,CLM_0000003,CTR_002653,2024-10-25,2024-09-26,Fire,1762.45€,1030.68 EUR,Closed,EXP_001,Third_party
3,CLM_0000004,CTR_002271,2024-04-16,2024-05-27,Collision,3144.06 euros,2119.97€,Closed,NaN,Insured
4,CLM_0000005,CTR_000649,08/03/2025,2025-04-12,Collision,3715.18€,NaN,Expert_review,EXP_007,Third_party


In [14]:
vehicles.head()

,contract_id,brand,model,year,power,fuel_type,current_value,color,usage,previous_claims
0,CTR_000003,BMW,Serie1,"2,022.00",128 HP,Gasoline,29567.77€,Gray,Mixed,0.00
1,CTR_000005,Renault,Megane,"2,024.00",150 HP,Hybrid,14873.92€,Black,Personal,0.00
2,CTR_000012,Peugeot,208,"2,020.00",175,Hybrid,$8362.30,White,Professional,2.00
3,CTR_000015,Renault,Captur,"2,024.00",NaN,Electric,14849.30€,NaN,Mixed,1.00
4,CTR_000016,Renault,Megane,NaN,176hp,Gasoline,€14522.08,Blue,Personal,1.00


In [15]:
contracts["annual_premium"] = parse_monetary_series(
    contracts["annual_premium"]
)

claims["damage_amount"] = parse_monetary_series(
    claims["damage_amount"]
)

claims["indemnified_amount"] = parse_monetary_series(
    claims["indemnified_amount"]
)

vehicles["current_value"] = parse_monetary_series(
    vehicles["current_value"]
)

### Monetary Parsing Validation

After standardization, the cleaned monetary variables are compared with their raw counterparts.

The validation verifies that:

- All originally available monetary values remain available after parsing.
- No new missing values are introduced.
- Structurally missing indemnification values remain missing.
- Monetary columns are converted to numeric data types.

In [16]:
monetary_validation = pd.DataFrame(
    [
        {
            "dataset": "contracts",
            "column": "annual_premium",
            "raw_missing": contracts_raw["annual_premium"].isna().sum(),
            "clean_missing": contracts["annual_premium"].isna().sum(),
            "clean_dtype": contracts["annual_premium"].dtype
        },

        {
            "dataset": "claims",
            "column": "damage_amount",
            "raw_missing": claims_raw["damage_amount"].isna().sum(),
            "clean_missing": claims["damage_amount"].isna().sum(),
            "clean_dtype": claims["damage_amount"].dtype
        },

        {
            "dataset": "claims",
            "column": "indemnified_amount",
            "raw_missing": claims_raw["indemnified_amount"].isna().sum(),
            "clean_missing": claims["indemnified_amount"].isna().sum(),
            "clean_dtype": claims["indemnified_amount"].dtype
        },

        {
            "dataset": "vehicles",
            "column": "current_value",
            "raw_missing": vehicles_raw["current_value"].isna().sum(),
            "clean_missing": vehicles["current_value"].isna().sum(),
            "clean_dtype": vehicles["current_value"].dtype
        },        
    ]
)

monetary_validation

,dataset,column,raw_missing,clean_missing,clean_dtype
0,contracts,annual_premium,0,0,Float64
1,claims,damage_amount,0,0,Float64
2,claims,indemnified_amount,65,65,Float64
3,vehicles,current_value,0,0,Float64


In [17]:
monetary_validation["new_missing_values"] = (
    monetary_validation["clean_missing"]
    - monetary_validation["raw_missing"]
)

monetary_validation

,dataset,column,raw_missing,clean_missing,clean_dtype,new_missing_values
0,contracts,annual_premium,0,0,Float64,0
1,claims,damage_amount,0,0,Float64,0
2,claims,indemnified_amount,65,65,Float64,0
3,vehicles,current_value,0,0,Float64,0


In [18]:
pd.DataFrame(
    {
        "raw_annual_premium": contracts_raw["annual_premium"].head(10),
        "clean_annual_premium": contracts["annual_premium"].head(10),
    }
)

,raw_annual_premium,clean_annual_premium
0,1974.98€,"1,974.98"
1,€620.93,620.93
2,$1568.11,"1,568.11"
3,€1752.17,"1,752.17"
4,$977.59,977.59
5,549.89€,549.89
6,681.51€,681.51
7,579.32€,579.32
8,229.20€,229.20
9,312.56€,312.56


### Monetary Standardization Findings

All monetary variables were successfully converted to numeric values without introducing additional missing observations.

The cleaning process produced the following results:

- `annual_premium` was converted successfully with no missing values.
- `damage_amount` was converted successfully with no missing values.
- `indemnified_amount` preserved its 65 structurally missing observations.
- `current_value` was converted successfully with no missing values.
- Currency symbols, codes, and textual representations were removed while preserving the underlying numeric amounts.

The cleaned monetary variables are now suitable for numerical analysis, SQL storage, and future machine learning workflows.

## 3. Date Standardization

The raw datasets contain multiple date representations, including ISO, day-first, and month-first formats.

Some dates can be interpreted unambiguously, while others contain both day and month values below or equal to 12 and therefore support more than one valid interpretation.

To avoid silently assigning an incorrect date:

1. Unambiguous dates are parsed using explicit rules.
2. Ambiguous dates are identified separately.
3. Ambiguous contract and claim dates will later be resolved using business relationships between paired dates.

The raw date columns remain available in the `_raw` DataFrames for validation.

In [22]:
import re


def parse_unambiguous_date(value):
    """
    Parse a date only when its format can be interpreted
    unambiguously.

    Ambiguous slash-separated dates such as 08/03/2025
    are intentionally returned as NaT.
    """
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    # YYYY-MM-DD
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        return pd.to_datetime(
            value,
            format="%Y-%m-%d",
            errors="coerce",
        )

    # DD-MM-YYYY
    if re.fullmatch(r"\d{2}-\d{2}-\d{4}", value):
        return pd.to_datetime(
            value,
            format="%d-%m-%Y",
            errors="coerce",
        )

    # Slash-separated dates
    if re.fullmatch(r"\d{2}/\d{2}/\d{4}", value):
        first, second, year = map(
            int,
            value.split("/"),
        )

        # First value cannot be month -> DD/MM/YYYY
        if first > 12:
            return pd.to_datetime(
                value,
                format="%d/%m/%Y",
                errors="coerce",
            )

        # Second value cannot be month -> MM/DD/YYYY
        if second > 12:
            return pd.to_datetime(
                value,
                format="%m/%d/%Y",
                errors="coerce",
            )

        # Both are <= 12 -> ambiguous
        return pd.NaT

    return pd.NaT

In [23]:
date_test = pd.Series(
    [
        "2025-08-12",
        "26-11-2023",
        "26/11/2023",
        "04/17/2023",
        "08/03/2025",
    ]
)

pd.DataFrame(
    {
        "raw_date": date_test,
        "parsed_date": date_test.apply(
            parse_unambiguous_date
        ),
    }
)

,raw_date,parsed_date
0,2025-08-12,2025-08-12
1,26-11-2023,2023-11-26
2,26/11/2023,2023-11-26
3,04/17/2023,2023-04-17
4,08/03/2025,NaT


In [24]:
contracts["start_date_parsed"] = (
    contracts["start_date"]
    .apply(parse_unambiguous_date)
)

contracts["end_date_parsed"] = (
    contracts["end_date"]
    .apply(parse_unambiguous_date)
)

claims["occurrence_date_parsed"] = (
    claims["occurrence_date"]
    .apply(parse_unambiguous_date)
)

claims["declaration_date_parsed"] = (
    claims["declaration_date"]
    .apply(parse_unambiguous_date)
)

In [25]:
date_cleaning_summary = pd.DataFrame(
    [
        {
            "dataset": "contracts",
            "column": "start_date",
            "total_rows": len(contracts),
            "resolved": contracts["start_date_parsed"].notna().sum(),
            "unresolved": contracts["start_date_parsed"].isna().sum(),
        },
        {
            "dataset": "contracts",
            "column": "end_date",
            "total_rows": len(contracts),
            "resolved": contracts["end_date_parsed"].notna().sum(),
            "unresolved": contracts["end_date_parsed"].isna().sum(),
        },
        {
            "dataset": "claims",
            "column": "occurrence_date",
            "total_rows": len(claims),
            "resolved": claims["occurrence_date_parsed"].notna().sum(),
            "unresolved": claims["occurrence_date_parsed"].isna().sum(),
        },
        {
            "dataset": "claims",
            "column": "declaration_date",
            "total_rows": len(claims),
            "resolved": claims["declaration_date_parsed"].notna().sum(),
            "unresolved": claims["declaration_date_parsed"].isna().sum(),
        },
    ]
)

date_cleaning_summary["unresolved_pct"] = (
    date_cleaning_summary["unresolved"]
    / date_cleaning_summary["total_rows"]
    * 100
)

date_cleaning_summary

,dataset,column,total_rows,resolved,unresolved,unresolved_pct
0,contracts,start_date,15000,14407,593,3.95
1,contracts,end_date,15000,14414,586,3.91
2,claims,occurrence_date,155,148,7,4.52
3,claims,declaration_date,155,146,9,5.81


### Initial Date Parsing Findings

Most date values were successfully resolved using explicit and unambiguous parsing rules.

Approximately 94–96% of the observations across all date columns could be interpreted directly, while a relatively small proportion remained unresolved:

- 3.95% of contract start dates
- 3.91% of contract end dates
- 4.52% of claim occurrence dates
- 5.81% of claim declaration dates

These unresolved observations are intentionally preserved rather than automatically parsed because their raw representations may support more than one valid day-month interpretation.

The next step is to examine how unresolved dates occur within related date pairs before applying business-rule-based resolution.

In [26]:
contracts["start_date_unresolved"] = (
    contracts["start_date_parsed"].isna()
)

contracts["end_date_unresolved"] = (
    contracts["end_date_parsed"].isna()
)

contract_date_pair_summary = (
    contracts.groupby(
        [
            "start_date_unresolved",
            "end_date_unresolved"
        ]
    )
    .size()
    .rename("record_count")
    .reset_index()
)

contract_date_pair_summary

,start_date_unresolved,end_date_unresolved,record_count
0,False,False,13839
1,False,True,568
2,True,False,575
3,True,True,18


In [27]:
claims["occurrence_date_unresolved"] = (
    claims["occurrence_date_parsed"].isna()
)

claims["declaration_date_unresolved"] = (
    claims["declaration_date_parsed"].isna()
)

claim_date_pair_summary = (
    claims
    .groupby(
        [
            "occurrence_date_unresolved",
            "declaration_date_unresolved",
        ]
    )
    .size()
    .rename("record_count")
    .reset_index()
)

claim_date_pair_summary

,occurrence_date_unresolved,declaration_date_unresolved,record_count
0,False,False,140
1,False,True,8
2,True,False,6
3,True,True,1


### Contract Duration Reference

Before resolving ambiguous contract dates, contract durations are calculated for records where both the start and end dates were parsed unambiguously.

The observed duration distribution will be used as a business reference when evaluating alternative interpretations of ambiguous contract dates.

No ambiguous dates are modified at this stage.

In [28]:
resolved_contract_dates = contracts.loc[
    contracts["start_date_parsed"].notna()
    & contracts["end_date_parsed"].notna(),
    [
        "contract_id",
        "start_date_parsed",
        "end_date_parsed",
    ],
].copy()

resolved_contract_dates["contract_duration_days"] = (
    resolved_contract_dates["end_date_parsed"]
    - resolved_contract_dates["start_date_parsed"]
).dt.days

In [29]:
resolved_contract_dates["contract_duration_days"].describe()

count   13,839.00
mean       364.90
std         24.77
min        305.00
25%        347.00
50%        365.00
75%        383.00
max        425.00
Name: contract_duration_days, dtype: float64

In [30]:
resolved_contract_dates[
    "contract_duration_days"
].value_counts().head(15)

contract_duration_days
367    242
363    237
372    228
370    227
360    222
371    222
364    218
361    216
359    213
357    212
362    211
366    210
365    209
368    208
358    202
Name: count, dtype: int64

In [31]:
print(
    "Negative durations:",
    (resolved_contract_dates["contract_duration_days"] < 0).sum()
)

print(
    "Durations above 730 days:",
    (resolved_contract_dates["contract_duration_days"] > 730).sum()
)

Negative durations: 0
Durations above 730 days: 0


### Contract Duration Findings

Contracts with unambiguous start and end dates show a highly consistent duration pattern.

Key observations include:

- The median contract duration is 365 days.
- The mean contract duration is approximately 365 days.
- Observed durations range from 305 to 425 days.
- No negative contract durations were detected.
- No unusually long contracts exceeding two years were observed.

This provides a strong empirical reference for resolving ambiguous contract dates.

For ambiguous records, alternative date interpretations can be evaluated based on whether they produce a duration consistent with the observed contract-duration range.

### Ambiguous Contract Date Candidate Analysis

For contracts where only one of the two dates is ambiguous, both possible day-month interpretations are generated.

Each candidate is evaluated using the empirical contract-duration range observed among unambiguous records.

This step determines whether ambiguous dates can be resolved reliably before modifying the cleaned dataset.

In [32]:
def get_ambiguous_date_candidates(value):
    """
    Return both possible interpretations of an ambiguous
    DD/MM/YYYY or MM/DD/YYYY date.

    Example:
        08/03/2025
        -> 2025-03-08
        -> 2025-08-03
    """
    if pd.isna(value):
        return pd.NaT, pd.NaT

    value = str(value).strip()

    parts = value.split("/")

    if len(parts) != 3:
        return pd.NaT, pd.NaT

    first, second, year = map(int, parts)

    if first > 12 or second > 12:
        return pd.NaT, pd.NaT

    day_first = pd.Timestamp(
        year=year,
        month=second,
        day=first,
    )

    month_first = pd.Timestamp(
        year=year,
        month=first,
        day=second,
    )

    return day_first, month_first

In [33]:
ambiguous_start = contracts.loc[
    contracts["start_date_unresolved"]
    & ~contracts["end_date_unresolved"],
    [
        "contract_id",
        "start_date",
        "end_date_parsed",
    ],
].copy()

In [34]:
start_candidates = (
    ambiguous_start["start_date"]
    .apply(get_ambiguous_date_candidates)
)

ambiguous_start[
    ["start_candidate_1", "start_candidate_2"]
] = pd.DataFrame(
    start_candidates.tolist(),
    index=ambiguous_start.index,
)

In [35]:
ambiguous_start["duration_candidate_1"] = (
    ambiguous_start["end_date_parsed"]
    - ambiguous_start["start_candidate_1"]
).dt.days

ambiguous_start["duration_candidate_2"] = (
    ambiguous_start["end_date_parsed"]
    - ambiguous_start["start_candidate_2"]
).dt.days

In [36]:
MIN_CONTRACT_DAYS = 305
MAX_CONTRACT_DAYS = 425

ambiguous_start["candidate_1_valid"] = (
    ambiguous_start["duration_candidate_1"]
    .between(
        MIN_CONTRACT_DAYS,
        MAX_CONTRACT_DAYS,
    )
)

ambiguous_start["candidate_2_valid"] = (
    ambiguous_start["duration_candidate_2"]
    .between(
        MIN_CONTRACT_DAYS,
        MAX_CONTRACT_DAYS,
    )
)

In [37]:
start_candidate_summary = (
    ambiguous_start
    .groupby(
        [
            "candidate_1_valid",
            "candidate_2_valid",
        ]
    )
    .size()
    .rename("record_count")
    .reset_index()
)

start_candidate_summary

,candidate_1_valid,candidate_2_valid,record_count
0,False,True,196
1,True,False,206
2,True,True,173


### Ambiguous Start Date Candidate Findings

Among the 575 contracts with an ambiguous start date and an unambiguous end date:

- 206 records have only the first date interpretation within the observed contract-duration range.
- 196 records have only the second interpretation within the valid range.
- 173 records have two plausible interpretations.
- No record has both interpretations outside the expected duration range.

Therefore, 402 of the 575 ambiguous start dates (approximately 69.9%) can be resolved directly using the observed contract-duration range.

The remaining 173 records require an additional decision rule because both possible interpretations produce plausible contract durations.

In [38]:
ambiguous_start["candidate_1_distance_from_365"] = (
    ambiguous_start["duration_candidate_1"] - 365
).abs()

ambiguous_start["candidate_2_distance_from_365"] = (
    ambiguous_start["duration_candidate_2"] - 365
).abs()

In [39]:
both_valid_start = ambiguous_start.loc[
    ambiguous_start["candidate_1_valid"]
    & ambiguous_start["candidate_2_valid"]
].copy()

In [40]:
both_valid_start["preferred_candidate"] = np.select(
    [
        (
            both_valid_start["candidate_1_distance_from_365"]
            < both_valid_start["candidate_2_distance_from_365"]
        ),
        (
            both_valid_start["candidate_2_distance_from_365"]
            < both_valid_start["candidate_1_distance_from_365"]
        ),
    ],
    [
        "candidate_1",
        "candidate_2",
    ],
    default="tie",
)

both_valid_start["preferred_candidate"].value_counts()

preferred_candidate
candidate_1    86
candidate_2    45
tie            42
Name: count, dtype: int64

In [41]:
both_valid_start["distance_advantage_days"] = (
    both_valid_start[
        [
            "candidate_1_distance_from_365",
            "candidate_2_distance_from_365",
        ]
    ]
    .max(axis=1)
    -
    both_valid_start[
        [
            "candidate_1_distance_from_365",
            "candidate_2_distance_from_365",
        ]
    ]
    .min(axis=1)
)

both_valid_start["distance_advantage_days"].describe()

count   173.00
mean     18.05
std      15.57
min       0.00
25%       1.00
50%      21.00
75%      29.00
max      59.00
Name: distance_advantage_days, dtype: float64

### Contract Start Date Candidate Selection Findings

Among the 173 records where both date interpretations produced valid contract durations:

- 86 records favor the first interpretation.
- 45 records favor the second interpretation.
- 42 records result in an equal distance from the typical 365-day contract duration.

Combining the contract-duration validity rule with proximity to the median contract duration resolves an additional 131 records.

The remaining tie cases require inspection because some may represent dates where the day and month are identical, meaning both interpretations correspond to the same calendar date.

In [42]:
tie_start_dates = both_valid_start.loc[
    both_valid_start["preferred_candidate"].eq("tie"),
    [
        "contract_id",
        "start_date",
        "end_date_parsed",
        "start_candidate_1",
        "start_candidate_2",
        "duration_candidate_1",
        "duration_candidate_2",
    ],
].copy()

tie_start_dates.head(20)

,contract_id,start_date,end_date_parsed,start_candidate_1,start_candidate_2,duration_candidate_1,duration_candidate_2
326,CTR_000327,04/04/2025,2026-04-22,2025-04-04,2025-04-04,383,383
552,CTR_000553,06/06/2023,2024-06-28,2023-06-06,2023-06-06,388,388
707,CTR_000708,04/04/2023,2024-03-17,2023-04-04,2023-04-04,348,348
2017,CTR_002018,07/07/2024,2025-06-05,2024-07-07,2024-07-07,333,333
2085,CTR_002086,12/12/2024,2025-12-18,2024-12-12,2024-12-12,371,371
2571,CTR_002572,08/09/2025,2026-08-24,2025-09-08,2025-08-09,350,380
2661,CTR_002662,11/11/2024,2025-10-31,2024-11-11,2024-11-11,354,354
2762,CTR_002763,10/10/2023,2024-11-11,2023-10-10,2023-10-10,398,398
2766,CTR_002767,03/03/2025,2026-03-15,2025-03-03,2025-03-03,377,377
3681,CTR_003682,09/09/2024,2025-08-03,2024-09-09,2024-09-09,328,328


In [43]:
tie_start_dates["same_candidate_date"] = (
    tie_start_dates["start_candidate_1"]
    == tie_start_dates["start_candidate_2"]
)

tie_start_dates["same_candidate_date"].value_counts()

same_candidate_date
True     40
False     2
Name: count, dtype: int64

In [44]:
tie_start_dates["start_date"].value_counts().head(20)

start_date
04/04/2025    3
03/03/2025    3
10/10/2024    3
02/02/2024    3
06/06/2023    2
11/11/2024    2
10/10/2023    2
08/08/2024    2
03/03/2023    2
03/03/2024    2
08/08/2025    2
04/04/2023    1
07/07/2024    1
12/12/2024    1
08/09/2025    1
09/09/2024    1
06/06/2025    1
01/01/2023    1
10/10/2025    1
01/01/2024    1
Name: count, dtype: int64

### Contract Start Date Tie Findings

Among the 42 records where both candidate interpretations were equally close to the typical 365-day contract duration:

- 40 records contain identical day and month values, such as `04/04/2025` or `12/12/2024`.
- For these records, both interpretations represent the same calendar date and therefore do not constitute true ambiguity.
- Only 2 contract records remain genuinely ambiguous, where the two candidate dates are different but produce equally plausible contract durations.

Therefore, 573 of the 575 initially ambiguous contract start dates can now be resolved using deterministic rules.

The remaining two records require additional inspection before a final date interpretation is assigned.

In [45]:
true_start_ties = tie_start_dates.loc[
    ~tie_start_dates["same_candidate_date"]
].copy()

true_start_ties

,contract_id,start_date,end_date_parsed,start_candidate_1,start_candidate_2,duration_candidate_1,duration_candidate_2,same_candidate_date
2571,CTR_002572,08/09/2025,2026-08-24,2025-09-08,2025-08-09,350,380,False
9854,CTR_009855,03/01/2024,2025-01-31,2024-01-03,2024-03-01,394,336,False


In [46]:
true_start_ties_detail = (
    true_start_ties
    .merge(
        contracts[
            [
                "contract_id",
                "product",
                "status",
                "annual_premium",
                "risk_zone",
                "channel",
            ]
        ],
        on = "contract_id",
        how = "left",
        validate = "one_to_one"
    )
)

true_start_ties_detail

,contract_id,start_date,end_date_parsed,start_candidate_1,start_candidate_2,duration_candidate_1,duration_candidate_2,same_candidate_date,product,status,annual_premium,risk_zone,channel
0,CTR_002572,08/09/2025,2026-08-24,2025-09-08,2025-08-09,350,380,False,Home,Suspended,319.48,Low,Broker
1,CTR_009855,03/01/2024,2025-01-31,2024-01-03,2024-03-01,394,336,False,Health,Cancelled,"1,600.42",Medium,Phone


### Remaining Ambiguous Contract Start Dates

Only two contract start dates remain genuinely ambiguous after applying the contract-duration validity and proximity rules.

For both records, the two possible date interpretations are equally distant from the overall median contract duration of 365 days.

Therefore, assigning either interpretation based solely on the overall duration distribution would be arbitrary.

As an additional validation step, contract-duration distributions will be examined separately by insurance product. This may provide a more specific business reference for resolving the remaining Home and Health contract dates.

In [47]:
resolved_contract_with_product = (
    resolved_contract_dates
    .merge(
        contracts[
            [
                "contract_id",
                "product",
            ]
        ],
        on="contract_id",
        how="left",
        validate="one_to_one",
    )
)

In [48]:
product_duration_summary = (
    resolved_contract_with_product
    .groupby("product")["contract_duration_days"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        max="max",
    )
    .round(2)
)

product_duration_summary

,count,mean,median,std,min,q25,q75,max
product,,,,,,,,
Auto,5852,364.79,365.00,24.90,305,347.00,382.00,424
Health,1543,363.68,363.00,24.36,305,347.00,381.00,423
Home,4102,365.19,365.00,24.62,305,348.00,383.00,425
Life,2342,365.48,366.00,24.98,305,347.00,384.00,425


### Product-Level Duration Findings

Contract-duration distributions are highly similar across insurance products.

Median durations range only from 363 to 366 days, and the interquartile ranges substantially overlap across Auto, Home, Life, and Health policies.

Therefore, product-specific duration patterns do not provide sufficient evidence to resolve the two remaining ambiguous contract start dates reliably.

Rather than assigning an arbitrary interpretation, these two dates will remain unresolved (`NaT`) in the cleaned date field while their original raw values are preserved.

This approach avoids introducing potentially incorrect information into the cleaned dataset.

In [49]:
ambiguous_start["resolved_start_date"] = pd.NaT

In [50]:
mask_candidate_1_only = (
    ambiguous_start["candidate_1_valid"]
    & ~ambiguous_start["candidate_2_valid"]
)

ambiguous_start.loc[
    mask_candidate_1_only,
    "resolved_start_date"
] = ambiguous_start.loc[
    mask_candidate_1_only,
    "start_candidate_1"
]

In [51]:
mask_candidate_2_only = (
    ~ambiguous_start["candidate_1_valid"]
    & ambiguous_start["candidate_2_valid"]
)

ambiguous_start.loc[
    mask_candidate_2_only,
    "resolved_start_date"
] = ambiguous_start.loc[
    mask_candidate_2_only,
    "start_candidate_2"
]

In [52]:
mask_candidate_1_closer = (
    ambiguous_start["candidate_1_valid"]
    & ambiguous_start["candidate_2_valid"]
    & (
        ambiguous_start["candidate_1_distance_from_365"]
        < ambiguous_start["candidate_2_distance_from_365"]
    )
)

ambiguous_start.loc[
    mask_candidate_1_closer,
    "resolved_start_date"
] = ambiguous_start.loc[
    mask_candidate_1_closer,
    "start_candidate_1"
]

In [53]:
mask_candidate_2_closer = (
    ambiguous_start["candidate_1_valid"]
    & ambiguous_start["candidate_2_valid"]
    & (
        ambiguous_start["candidate_2_distance_from_365"]
        < ambiguous_start["candidate_1_distance_from_365"]
    )
)

ambiguous_start.loc[
    mask_candidate_2_closer,
    "resolved_start_date"
] = ambiguous_start.loc[
    mask_candidate_2_closer,
    "start_candidate_2"
]

In [54]:
mask_same_date = (
    ambiguous_start["candidate_1_valid"]
    & ambiguous_start["candidate_2_valid"]
    & (
        ambiguous_start["start_candidate_1"]
        == ambiguous_start["start_candidate_2"]
    )
)

ambiguous_start.loc[
    mask_same_date,
    "resolved_start_date"
] = ambiguous_start.loc[
    mask_same_date,
    "start_candidate_1"
]

In [55]:
print(
    "Initially ambiguous start dates:",
    len(ambiguous_start)
)

print(
    "Resolved start dates:",
    ambiguous_start["resolved_start_date"].notna().sum()
)

print(
    "Still unresolved:",
    ambiguous_start["resolved_start_date"].isna().sum()
)

Initially ambiguous start dates: 575
Resolved start dates: 573
Still unresolved: 2


### Contract Start Date Resolution

Among the 575 contracts where the start date was ambiguous but the end date was already resolved, 573 start dates were successfully resolved using contract-duration rules.

Two records within this group remain genuinely ambiguous.

An additional 18 contracts have both their start and end dates unresolved. These records have not yet been processed because neither side currently provides a reliable reference date.

Therefore, after the first start-date resolution stage:

- 14,980 contract start dates are resolved.
- 20 contract start dates remain unresolved.
- Of these 20 records, 18 have both contract dates unresolved and 2 represent genuine start-date ambiguities.

In [58]:
remaining_start_unresolved = contracts.loc[
    contracts["start_date_parsed"].isna(),
    [
        "contract_id",
        "start_date",
        "end_date",
        "start_date_parsed",
        "end_date_parsed",
        "product",
    ],
].copy()

remaining_start_unresolved

,contract_id,start_date,end_date,start_date_parsed,end_date_parsed,product
1074,CTR_001075,03/02/2023,03/03/2024,NaT,NaT,Auto
1459,CTR_001460,05/09/2023,02/05/2024,NaT,NaT,Life
2096,CTR_002097,07/04/2023,12/08/2024,NaT,NaT,Auto
2571,CTR_002572,08/09/2025,2026-08-24,NaT,2026-08-24,Home
3885,CTR_003886,07/12/2024,12/06/2025,NaT,NaT,Home
4220,CTR_004221,02/08/2025,06/02/2026,NaT,NaT,Life
5100,CTR_005101,05/09/2024,10/08/2025,NaT,NaT,Life
6385,CTR_006386,11/09/2023,09/06/2024,NaT,NaT,Auto
6608,CTR_006609,05/05/2024,03/05/2025,NaT,NaT,Life
7588,CTR_007589,08/12/2023,08/11/2024,NaT,NaT,Home


In [59]:
remaining_start_unresolved.assign(
    end_date_also_unresolved=(
        remaining_start_unresolved["end_date_parsed"].isna()
    )
)["end_date_also_unresolved"].value_counts()

end_date_also_unresolved
True     18
False     2
Name: count, dtype: int64

### Ambiguous Contract End Date Resolution

Contracts with an ambiguous end date but a resolved start date are evaluated using the same empirical contract-duration rules applied to ambiguous start dates.

For each ambiguous end date:

- Both possible day-month interpretations are generated.
- The resulting contract duration is calculated.
- Candidate dates are evaluated against the observed valid contract-duration range of 305–425 days.
- Records with a single valid interpretation can be resolved deterministically.

No end dates are modified until the candidate analysis is validated.

In [60]:
ambiguous_end = contracts.loc[
    contracts["end_date_parsed"].isna()
    & contracts["start_date_parsed"].notna(),
    [
        "contract_id",
        "end_date",
        "start_date_parsed",
    ],
].copy()

print("Ambiguous end-date records:", len(ambiguous_end))

Ambiguous end-date records: 568


In [61]:
end_candidates = (
    ambiguous_end["end_date"]
    .apply(get_ambiguous_date_candidates)
)

ambiguous_end[
    ["end_candidate_1", "end_candidate_2"]
] = pd.DataFrame(
    end_candidates.tolist(),
    index=ambiguous_end.index,
)

In [62]:
ambiguous_end["duration_candidate_1"] = (
    ambiguous_end["end_candidate_1"]
    - ambiguous_end["start_date_parsed"]
).dt.days

ambiguous_end["duration_candidate_2"] = (
    ambiguous_end["end_candidate_2"]
    - ambiguous_end["start_date_parsed"]
).dt.days

In [63]:
ambiguous_end["candidate_1_valid"] = (
    ambiguous_end["duration_candidate_1"]
    .between(
        MIN_CONTRACT_DAYS,
        MAX_CONTRACT_DAYS,
    )
)

ambiguous_end["candidate_2_valid"] = (
    ambiguous_end["duration_candidate_2"]
    .between(
        MIN_CONTRACT_DAYS,
        MAX_CONTRACT_DAYS,
    )
)

In [64]:
end_candidate_summary = (
    ambiguous_end
    .groupby(
        [
            "candidate_1_valid",
            "candidate_2_valid",
        ]
    )
    .size()
    .rename("record_count")
    .reset_index()
)

end_candidate_summary

,candidate_1_valid,candidate_2_valid,record_count
0,False,True,209
1,True,False,192
2,True,True,167


### Ambiguous End Date Candidate Findings

Among the 568 contracts with an ambiguous end date and a resolved start date:

- 192 records have only the first date interpretation within the observed contract-duration range.
- 209 records have only the second interpretation within the valid range.
- 167 records have two plausible interpretations.
- No records have both interpretations outside the expected duration range.

Therefore, 401 of the 568 ambiguous end dates can be resolved directly using the observed contract-duration range.

The remaining 167 records require an additional decision rule because both interpretations produce plausible contract durations.

In [65]:
ambiguous_end["candidate_1_distance_from_365"] = (
    ambiguous_end["duration_candidate_1"] - 365
).abs()

ambiguous_end["candidate_2_distance_from_365"] = (
    ambiguous_end["duration_candidate_2"] - 365
).abs()

In [66]:
both_valid_end = ambiguous_end.loc[
    ambiguous_end["candidate_1_valid"]
    & ambiguous_end["candidate_2_valid"]
].copy()

In [67]:
both_valid_end["preferred_candidate"] = np.select(
    [
        (
            both_valid_end["candidate_1_distance_from_365"]
            < both_valid_end["candidate_2_distance_from_365"]
        ),
        (
            both_valid_end["candidate_2_distance_from_365"]
            < both_valid_end["candidate_1_distance_from_365"]
        ),
    ],
    [
        "candidate_1",
        "candidate_2",
    ],
    default="tie",
)

both_valid_end["preferred_candidate"].value_counts()

preferred_candidate
candidate_1    62
candidate_2    58
tie            47
Name: count, dtype: int64

In [68]:
both_valid_end["distance_advantage_days"] = (
    both_valid_end[
        [
            "candidate_1_distance_from_365",
            "candidate_2_distance_from_365",
        ]
    ].max(axis=1)
    -
    both_valid_end[
        [
            "candidate_1_distance_from_365",
            "candidate_2_distance_from_365",
        ]
    ].min(axis=1)
)

both_valid_end["distance_advantage_days"].describe()

count   167.00
mean     17.90
std      16.05
min       0.00
25%       0.00
50%      18.00
75%      29.00
max      59.00
Name: distance_advantage_days, dtype: float64

### Contract End Date Candidate Selection Findings

Among the 167 records where both end-date interpretations produced valid contract durations:

- 62 records favor the first interpretation.
- 58 records favor the second interpretation.
- 47 records result in an equal distance from the typical 365-day contract duration.

Combining the contract-duration validity rule with proximity to the median duration resolves an additional 120 end dates.

The remaining 47 tie cases require further inspection because some may contain identical day and month values, meaning that both candidate interpretations represent the same calendar date.

In [69]:
tie_end_dates = both_valid_end.loc[
    both_valid_end["preferred_candidate"].eq("tie"),
    [
        "contract_id",
        "end_date",
        "start_date_parsed",
        "end_candidate_1",
        "end_candidate_2",
        "duration_candidate_1",
        "duration_candidate_2",
    ],
].copy()

In [70]:
tie_end_dates["same_candidate_date"] = (
    tie_end_dates["end_candidate_1"]
    == tie_end_dates["end_candidate_2"]
)

tie_end_dates["same_candidate_date"].value_counts()

same_candidate_date
True     44
False     3
Name: count, dtype: int64

In [71]:
tie_end_dates[
    [
        "contract_id",
        "end_date",
        "end_candidate_1",
        "end_candidate_2",
        "duration_candidate_1",
        "duration_candidate_2",
        "same_candidate_date",
    ]
].head(20)

,contract_id,end_date,end_candidate_1,end_candidate_2,duration_candidate_1,duration_candidate_2,same_candidate_date
269,CTR_000270,08/08/2024,2024-08-08,2024-08-08,369,369,True
297,CTR_000298,04/04/2025,2025-04-04,2025-04-04,364,364,True
411,CTR_000412,10/10/2025,2025-10-10,2025-10-10,355,355,True
933,CTR_000934,02/02/2024,2024-02-02,2024-02-02,408,408,True
1376,CTR_001377,11/11/2024,2024-11-11,2024-11-11,333,333,True
1812,CTR_001813,12/12/2025,2025-12-12,2025-12-12,369,369,True
2195,CTR_002196,01/01/2024,2024-01-01,2024-01-01,366,366,True
3036,CTR_003037,01/01/2025,2025-01-01,2025-01-01,312,312,True
3468,CTR_003469,03/03/2025,2025-03-03,2025-03-03,361,361,True
4110,CTR_004111,03/03/2026,2026-03-03,2026-03-03,392,392,True


### Contract End Date Tie Findings

Among the 47 end-date records where both candidate interpretations were equally close to the typical 365-day contract duration:

- 44 records contain identical day and month values, meaning that both interpretations represent the same calendar date.
- Only 3 records remain genuinely ambiguous because the two candidate dates are different while producing equally plausible contract durations.

Therefore, 565 of the 568 initially ambiguous contract end dates can be resolved using deterministic rules.

The remaining three records require additional inspection before a final interpretation is assigned.

In [72]:
true_end_ties = tie_end_dates.loc[
    ~tie_end_dates["same_candidate_date"]
].copy()

true_end_ties

,contract_id,end_date,start_date_parsed,end_candidate_1,end_candidate_2,duration_candidate_1,duration_candidate_2,same_candidate_date
5161,CTR_005162,07/08/2025,2024-07-23,2025-08-07,2025-07-08,380,350,False
6100,CTR_006101,08/09/2025,2024-08-24,2025-09-08,2025-08-09,380,350,False
9953,CTR_009954,01/03/2024,2023-02-01,2024-03-01,2024-01-03,394,336,False


In [73]:
true_end_ties_detail = (
    true_end_ties
    .merge(
        contracts[
            [
                "contract_id",
                "product",
                "status",
                "annual_premium",
                "risk_zone",
                "channel",
            ]
        ],
        on="contract_id",
        how="left",
        validate="one_to_one",
    )
)

true_end_ties_detail[
    [
        "contract_id",
        "end_date",
        "start_date_parsed",
        "end_candidate_1",
        "end_candidate_2",
        "duration_candidate_1",
        "duration_candidate_2",
        "product",
        "status",
    ]
]

,contract_id,end_date,start_date_parsed,end_candidate_1,end_candidate_2,duration_candidate_1,duration_candidate_2,product,status
0,CTR_005162,07/08/2025,2024-07-23,2025-08-07,2025-07-08,380,350,Auto,Expired
1,CTR_006101,08/09/2025,2024-08-24,2025-09-08,2025-08-09,380,350,Auto,Cancelled
2,CTR_009954,01/03/2024,2023-02-01,2024-03-01,2024-01-03,394,336,Home,Renewed


### Remaining Ambiguous Contract End Dates

Three contract end dates remain genuinely ambiguous after applying the contract-duration validity and proximity rules.

For each of these records, the two possible date interpretations are equally distant from the typical contract duration.

Neither insurance product nor contract status provides sufficient evidence to select one interpretation reliably.

Therefore, these three end dates will remain unresolved (`NaT`) rather than being assigned arbitrarily.

This preserves data integrity and avoids introducing unsupported assumptions into the cleaned dataset.

In [74]:
ambiguous_end["resolved_end_date"] = pd.NaT

In [75]:
mask_candidate_1_only = (
    ambiguous_end["candidate_1_valid"]
    & ~ambiguous_end["candidate_2_valid"]
)

ambiguous_end.loc[
    mask_candidate_1_only,
    "resolved_end_date"
] = ambiguous_end.loc[
    mask_candidate_1_only,
    "end_candidate_1"
]

In [76]:
mask_candidate_2_only = (
    ~ambiguous_end["candidate_1_valid"]
    & ambiguous_end["candidate_2_valid"]
)

ambiguous_end.loc[
    mask_candidate_2_only,
    "resolved_end_date"
] = ambiguous_end.loc[
    mask_candidate_2_only,
    "end_candidate_2"
]

In [77]:
mask_candidate_1_closer = (
    ambiguous_end["candidate_1_valid"]
    & ambiguous_end["candidate_2_valid"]
    & (
        ambiguous_end["candidate_1_distance_from_365"]
        < ambiguous_end["candidate_2_distance_from_365"]
    )
)

ambiguous_end.loc[
    mask_candidate_1_closer,
    "resolved_end_date"
] = ambiguous_end.loc[
    mask_candidate_1_closer,
    "end_candidate_1"
]

In [78]:
mask_candidate_2_closer = (
    ambiguous_end["candidate_1_valid"]
    & ambiguous_end["candidate_2_valid"]
    & (
        ambiguous_end["candidate_2_distance_from_365"]
        < ambiguous_end["candidate_1_distance_from_365"]
    )
)

ambiguous_end.loc[
    mask_candidate_2_closer,
    "resolved_end_date"
] = ambiguous_end.loc[
    mask_candidate_2_closer,
    "end_candidate_2"
]

In [79]:
mask_same_date = (
    ambiguous_end["candidate_1_valid"]
    & ambiguous_end["candidate_2_valid"]
    & (
        ambiguous_end["end_candidate_1"]
        == ambiguous_end["end_candidate_2"]
    )
)

ambiguous_end.loc[
    mask_same_date,
    "resolved_end_date"
] = ambiguous_end.loc[
    mask_same_date,
    "end_candidate_1"
]

In [80]:
print(
    "Initially ambiguous end dates:",
    len(ambiguous_end)
)

print(
    "Resolved end dates:",
    ambiguous_end["resolved_end_date"].notna().sum()
)

print(
    "Still unresolved:",
    ambiguous_end["resolved_end_date"].isna().sum()
)

Initially ambiguous end dates: 568
Resolved end dates: 565
Still unresolved: 3


In [81]:
resolved_end_map = (
    ambiguous_end
    .set_index("contract_id")["resolved_end_date"]
)

contracts.loc[
    contracts["contract_id"].isin(resolved_end_map.index),
    "end_date_parsed",
] = (
    contracts.loc[
        contracts["contract_id"].isin(resolved_end_map.index),
        "contract_id",
    ]
    .map(resolved_end_map)
)

In [82]:
print(
    "Resolved end dates:",
    contracts["end_date_parsed"].notna().sum()
)

print(
    "Remaining unresolved end dates:",
    contracts["end_date_parsed"].isna().sum()
)

Resolved end dates: 14979
Remaining unresolved end dates: 21


### Contracts with Both Start and End Dates Ambiguous

Eighteen contracts contain ambiguous representations in both the start and end date fields.

Since neither date can initially serve as a reliable reference, all possible combinations of the two start-date and two end-date interpretations are evaluated.

For each contract, up to four possible date combinations are considered.

A candidate combination is considered plausible if the resulting contract duration falls within the empirically observed range of 305 to 425 days.

No dates are modified until the candidate combinations have been evaluated.

In [83]:
both_dates_ambiguous = contracts.loc[
    contracts["start_date_parsed"].isna()
    & contracts["end_date_parsed"].isna(),
    [
        "contract_id",
        "start_date",
        "end_date",
        "product",
    ],
].copy()

print(
    "Contracts with both dates unresolved:",
    len(both_dates_ambiguous)
)

Contracts with both dates unresolved: 18


In [84]:
start_candidates = (
    both_dates_ambiguous["start_date"]
    .apply(get_ambiguous_date_candidates)
)

both_dates_ambiguous[
    ["start_candidate_1", "start_candidate_2"]
] = pd.DataFrame(
    start_candidates.tolist(),
    index=both_dates_ambiguous.index,
)


end_candidates = (
    both_dates_ambiguous["end_date"]
    .apply(get_ambiguous_date_candidates)
)

both_dates_ambiguous[
    ["end_candidate_1", "end_candidate_2"]
] = pd.DataFrame(
    end_candidates.tolist(),
    index=both_dates_ambiguous.index,
)

In [85]:
both_dates_ambiguous["duration_11"] = (
    both_dates_ambiguous["end_candidate_1"]
    - both_dates_ambiguous["start_candidate_1"]
).dt.days

both_dates_ambiguous["duration_12"] = (
    both_dates_ambiguous["end_candidate_2"]
    - both_dates_ambiguous["start_candidate_1"]
).dt.days

both_dates_ambiguous["duration_21"] = (
    both_dates_ambiguous["end_candidate_1"]
    - both_dates_ambiguous["start_candidate_2"]
).dt.days

both_dates_ambiguous["duration_22"] = (
    both_dates_ambiguous["end_candidate_2"]
    - both_dates_ambiguous["start_candidate_2"]
).dt.days

In [86]:
duration_columns = [
    "duration_11",
    "duration_12",
    "duration_21",
    "duration_22",
]

for column in duration_columns:
    both_dates_ambiguous[f"{column}_valid"] = (
        both_dates_ambiguous[column]
        .between(
            MIN_CONTRACT_DAYS,
            MAX_CONTRACT_DAYS,
        )
    )

In [87]:
valid_columns = [
    "duration_11_valid",
    "duration_12_valid",
    "duration_21_valid",
    "duration_22_valid",
]

both_dates_ambiguous["valid_combination_count"] = (
    both_dates_ambiguous[valid_columns]
    .sum(axis=1)
)

both_dates_ambiguous[
    "valid_combination_count"
].value_counts().sort_index()

valid_combination_count
1     6
2    10
4     2
Name: count, dtype: int64

In [88]:
both_dates_ambiguous[
    [
        "contract_id",
        "start_date",
        "end_date",
        "duration_11",
        "duration_12",
        "duration_21",
        "duration_22",
        "valid_combination_count",
    ]
]

,contract_id,start_date,end_date,duration_11,duration_12,duration_21,duration_22,valid_combination_count
1074,CTR_001075,03/02/2023,03/03/2024,394,394,367,367,4
1459,CTR_001460,05/09/2023,02/05/2024,240,153,359,272,1
2096,CTR_002097,07/04/2023,12/08/2024,493,611,405,523,1
3885,CTR_003886,07/12/2024,12/06/2025,187,364,335,512,2
4220,CTR_004221,02/08/2025,06/02/2026,188,304,363,479,1
5100,CTR_005101,05/09/2024,10/08/2025,339,398,458,517,2
6385,CTR_006386,11/09/2023,09/06/2024,272,361,213,302,1
6608,CTR_006609,05/05/2024,03/05/2025,363,304,363,304,2
7588,CTR_007589,08/12/2023,08/11/2024,336,247,454,365,2
8258,CTR_008259,04/10/2024,04/03/2025,151,181,328,358,2


### Both-Date Ambiguity Findings

All 18 contracts with both start and end dates unresolved have at least one date combination that falls within the empirically observed contract-duration range of 305 to 425 days.

The candidate analysis shows that:

- 6 contracts have exactly one valid date combination.
- 10 contracts have two valid combinations.
- 2 contracts have four valid combinations.
- No contract has zero valid combinations.

The six contracts with a single valid combination can therefore be resolved deterministically.

Contracts with multiple valid combinations require an additional selection criterion. As in the previous date-resolution steps, proximity to the typical 365-day contract duration will be evaluated next.

In [89]:
duration_map = {
    "duration_11": ("start_candidate_1", "end_candidate_1"),
    "duration_12": ("start_candidate_1", "end_candidate_2"),
    "duration_21": ("start_candidate_2", "end_candidate_1"),
    "duration_22": ("start_candidate_2", "end_candidate_2"),
}

In [90]:
for duration_col in duration_columns:
    valid_col = f"{duration_col}_valid"
    distance_col = f"{duration_col}_distance"

    both_dates_ambiguous[distance_col] = np.where(
        both_dates_ambiguous[valid_col],
        (both_dates_ambiguous[duration_col] - 365).abs(),
        np.nan,
    )

In [91]:
distance_columns = [
    "duration_11_distance",
    "duration_12_distance",
    "duration_21_distance",
    "duration_22_distance",
]

both_dates_ambiguous["min_distance_from_365"] = (
    both_dates_ambiguous[distance_columns]
    .min(axis=1)
)

In [92]:
both_dates_ambiguous["best_candidate_count"] = (
    both_dates_ambiguous[distance_columns]
    .eq(
        both_dates_ambiguous["min_distance_from_365"],
        axis=0,
    )
    .sum(axis=1)
)

both_dates_ambiguous[
    "best_candidate_count"
].value_counts().sort_index()

best_candidate_count
1    12
2     6
Name: count, dtype: int64

In [93]:
both_dates_ambiguous[
    [
        "contract_id",
        "start_date",
        "end_date",
        "duration_11",
        "duration_12",
        "duration_21",
        "duration_22",
        "valid_combination_count",
        "min_distance_from_365",
        "best_candidate_count",
    ]
]

,contract_id,start_date,end_date,duration_11,duration_12,duration_21,duration_22,valid_combination_count,min_distance_from_365,best_candidate_count
1074,CTR_001075,03/02/2023,03/03/2024,394,394,367,367,4,2.00,2
1459,CTR_001460,05/09/2023,02/05/2024,240,153,359,272,1,6.00,1
2096,CTR_002097,07/04/2023,12/08/2024,493,611,405,523,1,40.00,1
3885,CTR_003886,07/12/2024,12/06/2025,187,364,335,512,2,1.00,1
4220,CTR_004221,02/08/2025,06/02/2026,188,304,363,479,1,2.00,1
5100,CTR_005101,05/09/2024,10/08/2025,339,398,458,517,2,26.00,1
6385,CTR_006386,11/09/2023,09/06/2024,272,361,213,302,1,4.00,1
6608,CTR_006609,05/05/2024,03/05/2025,363,304,363,304,2,2.00,2
7588,CTR_007589,08/12/2023,08/11/2024,336,247,454,365,2,0.00,1
8258,CTR_008259,04/10/2024,04/03/2025,151,181,328,358,2,7.00,1


### Unique Best Date-Pair Validation

Some contracts appear to have multiple equally preferred date combinations because one of the ambiguous raw dates produces identical calendar dates under both interpretations.

For example, a value such as `03/03/2024` generates the same date regardless of whether it is interpreted as day-first or month-first.

Therefore, the number of preferred combinations may overstate the true level of ambiguity.

To avoid counting identical date pairs more than once, all candidate start-end combinations are converted into a long-format table and duplicate calendar-date pairs are removed before identifying the final best candidate.

In [94]:
candidate_rows = []

for idx, row in both_dates_ambiguous.iterrows():
    combinations = [
        (
            "11",
            row["start_candidate_1"],
            row["end_candidate_1"],
            row["duration_11"],
        ),
        (
            "12",
            row["start_candidate_1"],
            row["end_candidate_2"],
            row["duration_12"],
        ),
        (
            "21",
            row["start_candidate_2"],
            row["end_candidate_1"],
            row["duration_21"],
        ),
        (
            "22",
            row["start_candidate_2"],
            row["end_candidate_2"],
            row["duration_22"],
        ),
    ]

    for combination, start_candidate, end_candidate, duration in combinations:
        candidate_rows.append(
            {
                "contract_id": row["contract_id"],
                "combination": combination,
                "start_candidate": start_candidate,
                "end_candidate": end_candidate,
                "duration_days": duration,
            }
        )

contract_date_candidates = pd.DataFrame(candidate_rows)

In [95]:
contract_date_candidates["valid"] = (
    contract_date_candidates["duration_days"]
    .between(
        MIN_CONTRACT_DAYS,
        MAX_CONTRACT_DAYS,
    )
)

valid_contract_date_candidates = (
    contract_date_candidates
    .loc[contract_date_candidates["valid"]]
    .copy()
)

In [96]:
unique_contract_date_candidates = (
    valid_contract_date_candidates
    .drop_duplicates(
        subset=[
            "contract_id",
            "start_candidate",
            "end_candidate",
        ]
    )
    .copy()
)

In [97]:
unique_contract_date_candidates["distance_from_365"] = (
    unique_contract_date_candidates["duration_days"] - 365
).abs()

In [98]:
unique_contract_date_candidates["min_distance"] = (
    unique_contract_date_candidates
    .groupby("contract_id")["distance_from_365"]
    .transform("min")
)

In [99]:
best_contract_date_candidates = (
    unique_contract_date_candidates
    .loc[
        unique_contract_date_candidates["distance_from_365"]
        == unique_contract_date_candidates["min_distance"]
    ]
    .copy()
)

In [100]:
unique_best_candidate_summary = (
    best_contract_date_candidates
    .groupby("contract_id")
    .size()
    .value_counts()
    .sort_index()
)

unique_best_candidate_summary

1    16
2     2
Name: count, dtype: int64

In [101]:
genuine_date_ties = (
    best_contract_date_candidates
    .groupby("contract_id")
    .filter(lambda x: len(x) > 1)
)

genuine_date_ties[
    [
        "contract_id",
        "start_candidate",
        "end_candidate",
        "duration_days",
        "distance_from_365",
    ]
].sort_values(
    ["contract_id", "start_candidate"]
)

,contract_id,start_candidate,end_candidate,duration_days,distance_from_365
41,CTR_008636,2024-03-07,2025-03-07,365,0
42,CTR_008636,2024-07-03,2025-07-03,365,0
66,CTR_014293,2025-05-07,2026-05-07,365,0
65,CTR_014293,2025-07-05,2026-07-05,365,0


### Final Resolution of Contracts with Both Dates Ambiguous

After removing duplicate candidate date pairs, 16 of the 18 contracts with both dates initially ambiguous have a single best interpretation.

Two contracts remain genuinely ambiguous:

- `CTR_008636`
- `CTR_014293`

For both records, two distinct start-end date combinations produce exactly 365-day contract durations.

Since no available business rule provides sufficient evidence to prefer one interpretation over the other, these dates will remain unresolved rather than being assigned arbitrarily.

This leaves 16 contracts that can be resolved deterministically and 2 contracts intentionally preserved as unresolved.

In [102]:
best_candidate_counts = (
    best_contract_date_candidates
    .groupby("contract_id")
    .size()
)

single_best_contract_ids = (
    best_candidate_counts[
        best_candidate_counts == 1
    ]
    .index
)

resolved_both_dates = (
    best_contract_date_candidates
    .loc[
        best_contract_date_candidates["contract_id"]
        .isin(single_best_contract_ids)
    ]
    .copy()
)

print(
    "Contracts resolved from both-date ambiguity:",
    len(resolved_both_dates)
)

Contracts resolved from both-date ambiguity: 16


In [103]:
resolved_both_start_map = (
    resolved_both_dates
    .set_index("contract_id")["start_candidate"]
)

resolved_both_end_map = (
    resolved_both_dates
    .set_index("contract_id")["end_candidate"]
)

In [104]:
contracts.loc[
    contracts["contract_id"].isin(
        resolved_both_start_map.index
    ),
    "start_date_parsed",
] = (
    contracts.loc[
        contracts["contract_id"].isin(
            resolved_both_start_map.index
        ),
        "contract_id",
    ]
    .map(resolved_both_start_map)
)

In [105]:
contracts.loc[
    contracts["contract_id"].isin(
        resolved_both_end_map.index
    ),
    "end_date_parsed",
] = (
    contracts.loc[
        contracts["contract_id"].isin(
            resolved_both_end_map.index
        ),
        "contract_id",
    ]
    .map(resolved_both_end_map)
)

In [106]:
print(
    "Resolved start dates:",
    contracts["start_date_parsed"].notna().sum()
)

print(
    "Unresolved start dates:",
    contracts["start_date_parsed"].isna().sum()
)

print()

print(
    "Resolved end dates:",
    contracts["end_date_parsed"].notna().sum()
)

print(
    "Unresolved end dates:",
    contracts["end_date_parsed"].isna().sum()
)

Resolved start dates: 14996
Unresolved start dates: 4

Resolved end dates: 14995
Unresolved end dates: 5


In [107]:
remaining_contract_date_issues = contracts.loc[
    contracts["start_date_parsed"].isna()
    | contracts["end_date_parsed"].isna(),
    [
        "contract_id",
        "start_date",
        "end_date",
        "start_date_parsed",
        "end_date_parsed",
        "product",
        "status",
    ],
].copy()

remaining_contract_date_issues

,contract_id,start_date,end_date,start_date_parsed,end_date_parsed,product,status
2571,CTR_002572,08/09/2025,2026-08-24,NaT,2026-08-24,Home,Suspended
5161,CTR_005162,2024-07-23,07/08/2025,2024-07-23,NaT,Auto,Expired
6100,CTR_006101,2024-08-24,08/09/2025,2024-08-24,NaT,Auto,Cancelled
8635,CTR_008636,07/03/2024,03/07/2025,NaT,NaT,Auto,Cancelled
9854,CTR_009855,03/01/2024,2025-01-31,NaT,2025-01-31,Health,Cancelled
9953,CTR_009954,2023-02-01,01/03/2024,2023-02-01,NaT,Home,Renewed
14292,CTR_014293,05/07/2025,07/05/2026,NaT,NaT,Health,Suspended


### Contract Date Cleaning Findings

Contract dates were standardized using explicit parsing rules and empirical business constraints derived from the observed contract-duration distribution.

The cleaning process successfully resolved nearly all initially ambiguous date representations.

Final results:

- 14,996 of 15,000 contract start dates were resolved.
- 14,995 of 15,000 contract end dates were resolved.
- Only 7 contracts contain at least one unresolved date field.

Among the remaining records:

- 2 contracts contain only an unresolved start date.
- 3 contracts contain only an unresolved end date.
- 2 contracts contain unresolved values for both dates.

These observations remain unresolved because multiple date interpretations are equally plausible and the available data does not provide sufficient evidence to select one interpretation reliably.

Rather than introducing arbitrary assumptions, the unresolved values are preserved as `NaT`.

This results in a contract-date resolution rate above 99.9%.

## 4. Claim Date Standardization

Claim occurrence and declaration dates contain the same mixed-format date representations observed in the contract data.

However, claim dates follow an additional chronological business rule:

> A claim declaration should not precede the occurrence of the insured event.

Before resolving ambiguous claim dates, records where both dates were parsed unambiguously are used to examine the typical delay between occurrence and declaration.

This empirical distribution will provide a reference for evaluating alternative date interpretations.

In [108]:
resolved_claim_dates = claims.loc[
    claims["occurrence_date_parsed"].notna()
    & claims["declaration_date_parsed"].notna(),
    [
        "claim_id",
        "occurrence_date",
        "declaration_date",
        "occurrence_date_parsed",
        "declaration_date_parsed",
    ],
].copy()

In [109]:
resolved_claim_dates["declaration_lag_days"] = (
    resolved_claim_dates["declaration_date_parsed"]
    - resolved_claim_dates["occurrence_date_parsed"]
).dt.days

In [110]:
print(
    "Claims with both dates resolved:",
    len(resolved_claim_dates)
)

print(
    "Declaration before occurrence:",
    (
        resolved_claim_dates["declaration_lag_days"] < 0
    ).sum()
)

print(
    "Declaration on occurrence date:",
    (
        resolved_claim_dates["declaration_lag_days"] == 0
    ).sum()
)

print(
    "Declaration after occurrence:",
    (
        resolved_claim_dates["declaration_lag_days"] > 0
    ).sum()
)

Claims with both dates resolved: 140
Declaration before occurrence: 57
Declaration on occurrence date: 5
Declaration after occurrence: 78


In [111]:
valid_claim_lags = resolved_claim_dates.loc[
    resolved_claim_dates["declaration_lag_days"] >= 0,
    "declaration_lag_days",
]

valid_claim_lags.describe()

count   83.00
mean    21.31
std     16.85
min      0.00
25%      7.00
50%     17.00
75%     34.00
max     62.00
Name: declaration_lag_days, dtype: float64

In [112]:
valid_claim_lags.sort_values(
    ascending=False
).head(15)

126    62
24     54
131    51
147    50
53     50
7      50
50     49
20     49
47     48
21     46
103    46
40     44
112    43
27     42
3      41
Name: declaration_lag_days, dtype: int64

### Claim Declaration Lag Findings

Among the 140 claim records where both occurrence and declaration dates were parsed unambiguously:

- 78 claims were declared after the occurrence date.
- 5 claims were declared on the same date as the occurrence.
- 57 claims contain a declaration date earlier than the occurrence date.

Therefore, the chronological inconsistency observed during data understanding is not caused solely by ambiguous date formats.

For chronologically valid claims, the declaration delay follows a relatively compact distribution:

- Median delay: 17 days
- Mean delay: approximately 21 days
- Interquartile range: 7–34 days
- Maximum observed delay: 62 days

This provides a useful empirical reference for resolving ambiguous claim dates.

However, the 57 chronologically inconsistent records must be investigated separately before any correction rule is applied.

In [113]:
invalid_claim_lags = resolved_claim_dates.loc[
    resolved_claim_dates["declaration_lag_days"] < 0
].copy()

invalid_claim_lags["absolute_lag_days"] = (
    invalid_claim_lags["declaration_lag_days"].abs()
)

In [114]:
invalid_claim_lags["declaration_lag_days"].describe()

count    57.00
mean    -21.26
std      14.07
min     -55.00
25%     -30.00
50%     -22.00
75%      -9.00
max      -1.00
Name: declaration_lag_days, dtype: float64

In [115]:
invalid_claim_lags["absolute_lag_days"].describe()

count   57.00
mean    21.26
std     14.07
min      1.00
25%      9.00
50%     22.00
75%     30.00
max     55.00
Name: absolute_lag_days, dtype: float64

In [116]:
invalid_claim_lags[
    [
        "claim_id",
        "occurrence_date",
        "declaration_date",
        "occurrence_date_parsed",
        "declaration_date_parsed",
        "declaration_lag_days",
    ]
].head(15)

,claim_id,occurrence_date,declaration_date,occurrence_date_parsed,declaration_date_parsed,declaration_lag_days
0,CLM_0000001,26-11-2023,2023-10-02,2023-11-26,2023-10-02,-55
2,CLM_0000003,2024-10-25,2024-09-26,2024-10-25,2024-09-26,-29
8,CLM_0000009,2025-07-22,2025-06-16,2025-07-22,2025-06-16,-36
10,CLM_0000011,2025-06-11,2025-06-10,2025-06-11,2025-06-10,-1
11,CLM_0000012,2023-09-04,2023-07-31,2023-09-04,2023-07-31,-35
12,CLM_0000013,2025-08-12,2025-06-28,2025-08-12,2025-06-28,-45
17,CLM_0000018,2025-04-01,2025-03-09,2025-04-01,2025-03-09,-23
18,CLM_0000019,2024-07-18,2024-06-17,2024-07-18,2024-06-17,-31
29,CLM_0000030,2025-06-08,03-06-2025,2025-06-08,2025-06-03,-5
32,CLM_0000033,2024-03-08,2024-03-07,2024-03-08,2024-03-07,-1


### Chronologically Inconsistent Claim Date Findings

The 57 claims with a declaration date earlier than the occurrence date show a highly structured pattern.

The absolute values of the negative declaration lags range from 1 to 55 days, with a mean of approximately 21 days. This distribution is remarkably similar to the valid declaration-lag distribution, which has a mean of approximately 21 days and a maximum of 62 days.

In addition, several affected records use unambiguous date formats, confirming that the issue cannot be explained solely by date parsing.

This pattern suggests that the occurrence and declaration dates may have been reversed for a subset of claim records.

Before applying a correction, a temporary swap will be simulated and the resulting declaration-lag distribution will be validated against the chronologically valid claims.

In [117]:
swapped_claim_dates = invalid_claim_lags.copy()

swapped_claim_dates["corrected_occurrence_date"] = (
    swapped_claim_dates["declaration_date_parsed"]
)

swapped_claim_dates["corrected_declaration_date"] = (
    swapped_claim_dates["occurrence_date_parsed"]
)

swapped_claim_dates["corrected_lag_days"] = (
    swapped_claim_dates["corrected_declaration_date"]
    - swapped_claim_dates["corrected_occurrence_date"]
).dt.days

In [118]:
swapped_claim_dates["corrected_lag_days"].describe()

count   57.00
mean    21.26
std     14.07
min      1.00
25%      9.00
50%     22.00
75%     30.00
max     55.00
Name: corrected_lag_days, dtype: float64

In [119]:
print(
    "Negative corrected lags:",
    (
        swapped_claim_dates["corrected_lag_days"] < 0
    ).sum()
)

print(
    "Corrected lags above 62 days:",
    (
        swapped_claim_dates["corrected_lag_days"] > 62
    ).sum()
)

print(
    "Corrected lag range:",
    (
        swapped_claim_dates["corrected_lag_days"].min(),
        swapped_claim_dates["corrected_lag_days"].max(),
    )
)

Negative corrected lags: 0
Corrected lags above 62 days: 0
Corrected lag range: (np.int64(1), np.int64(55))


In [120]:
claim_lag_comparison = pd.DataFrame(
    {
        "valid_claims": valid_claim_lags.describe(),
        "swapped_claims": (
            swapped_claim_dates["corrected_lag_days"]
            .describe()
        ),
    }
)

claim_lag_comparison

,valid_claims,swapped_claims
count,83.00,57.00
mean,21.31,21.26
std,16.85,14.07
min,0.00,1.00
25%,7.00,9.00
50%,17.00,22.00
75%,34.00,30.00
max,62.00,55.00


### Claim Date Reversal Findings

The temporary swap validation provides strong evidence that the occurrence and declaration dates were reversed for 57 claim records.

After swapping the two dates:

- No negative declaration lags remain.
- No corrected declaration lag exceeds the observed maximum of 62 days.
- Corrected declaration lags range from 1 to 55 days.
- The mean corrected lag is 21.26 days, almost identical to the 21.31-day mean observed among originally valid claim records.

The corrected lag distribution closely matches the distribution of chronologically valid claims.

Therefore, the 57 affected records will be corrected by swapping their parsed occurrence and declaration dates.

A dedicated flag will be retained to document which records were modified during the cleaning process.

In [121]:
swap_claim_date_idx = invalid_claim_lags.index

claims["claim_dates_swapped"] = False

claims.loc[
    swap_claim_date_idx,
    "claim_dates_swapped"
] = True

In [122]:
original_occurrence = claims.loc[
    swap_claim_date_idx,
    "occurrence_date_parsed"
].copy()

original_declaration = claims.loc[
    swap_claim_date_idx,
    "declaration_date_parsed"
].copy()

claims.loc[
    swap_claim_date_idx,
    "occurrence_date_parsed"
] = original_declaration

claims.loc[
    swap_claim_date_idx,
    "declaration_date_parsed"
] = original_occurrence

In [123]:
claims["claim_dates_swapped"].value_counts()

claim_dates_swapped
False    98
True     57
Name: count, dtype: int64

In [124]:
claim_date_validation = claims.loc[
    claims["occurrence_date_parsed"].notna()
    & claims["declaration_date_parsed"].notna()
].copy()

claim_date_validation["declaration_lag_days"] = (
    claim_date_validation["declaration_date_parsed"]
    - claim_date_validation["occurrence_date_parsed"]
).dt.days

In [125]:
print(
    "Claims with both dates resolved:",
    len(claim_date_validation)
)

print(
    "Declaration before occurrence:",
    (
        claim_date_validation["declaration_lag_days"] < 0
    ).sum()
)

print(
    "Declaration on occurrence date:",
    (
        claim_date_validation["declaration_lag_days"] == 0
    ).sum()
)

print(
    "Declaration after occurrence:",
    (
        claim_date_validation["declaration_lag_days"] > 0
    ).sum()
)

Claims with both dates resolved: 140
Declaration before occurrence: 0
Declaration on occurrence date: 5
Declaration after occurrence: 135


In [126]:
claim_date_validation["declaration_lag_days"].describe()

count   140.00
mean     21.29
std      15.72
min       0.00
25%       7.75
50%      19.00
75%      33.25
max      62.00
Name: declaration_lag_days, dtype: float64

### Claim Date Swap Validation

After correcting the 57 chronologically inconsistent claim records by swapping their parsed occurrence and declaration dates:

- No declaration dates occur before the corresponding occurrence dates.
- 5 claims were declared on the occurrence date.
- 135 claims were declared after the occurrence date.
- Declaration delays range from 0 to 62 days.
- The median declaration delay is 19 days.
- The mean declaration delay is approximately 21 days.

The resulting distribution is consistent with the previously observed valid claim records.

This confirms that the date swap provides a plausible and internally consistent correction for the affected claims.

### Ambiguous Claim Occurrence Dates

Claims with an ambiguous occurrence date but a resolved declaration date are evaluated using the observed declaration-delay distribution.

For each ambiguous occurrence date, both possible date interpretations are generated.

A candidate occurrence date is considered plausible when:

- It does not occur after the declaration date.
- The resulting declaration lag falls within the empirically observed range of 0 to 62 days.

No dates are modified until the candidate interpretations have been evaluated.

In [127]:
ambiguous_occurrence = claims.loc[
    claims["occurrence_date_parsed"].isna()
    & claims["declaration_date_parsed"].notna(),
    [
        "claim_id",
        "occurrence_date",
        "declaration_date_parsed",
    ],
].copy()

print(
    "Claims with ambiguous occurrence date:",
    len(ambiguous_occurrence)
)

Claims with ambiguous occurrence date: 6


In [128]:
occurrence_candidates = (
    ambiguous_occurrence["occurrence_date"]
    .apply(get_ambiguous_date_candidates)
)

ambiguous_occurrence[
    [
        "occurrence_candidate_1",
        "occurrence_candidate_2",
    ]
] = pd.DataFrame(
    occurrence_candidates.tolist(),
    index=ambiguous_occurrence.index,
)

In [129]:
ambiguous_occurrence["lag_candidate_1"] = (
    ambiguous_occurrence["declaration_date_parsed"]
    - ambiguous_occurrence["occurrence_candidate_1"]
).dt.days

ambiguous_occurrence["lag_candidate_2"] = (
    ambiguous_occurrence["declaration_date_parsed"]
    - ambiguous_occurrence["occurrence_candidate_2"]
).dt.days

In [130]:
MIN_CLAIM_LAG_DAYS = 0
MAX_CLAIM_LAG_DAYS = 62

ambiguous_occurrence["candidate_1_valid"] = (
    ambiguous_occurrence["lag_candidate_1"]
    .between(
        MIN_CLAIM_LAG_DAYS,
        MAX_CLAIM_LAG_DAYS,
    )
)

ambiguous_occurrence["candidate_2_valid"] = (
    ambiguous_occurrence["lag_candidate_2"]
    .between(
        MIN_CLAIM_LAG_DAYS,
        MAX_CLAIM_LAG_DAYS,
    )
)

In [131]:
occurrence_candidate_summary = (
    ambiguous_occurrence
    .groupby(
        [
            "candidate_1_valid",
            "candidate_2_valid",
        ]
    )
    .size()
    .rename("record_count")
    .reset_index()
)

occurrence_candidate_summary

,candidate_1_valid,candidate_2_valid,record_count
0,False,False,2
1,False,True,2
2,True,False,2


In [132]:
ambiguous_occurrence[
    [
        "claim_id",
        "occurrence_date",
        "declaration_date_parsed",
        "occurrence_candidate_1",
        "occurrence_candidate_2",
        "lag_candidate_1",
        "lag_candidate_2",
        "candidate_1_valid",
        "candidate_2_valid",
    ]
]

,claim_id,occurrence_date,declaration_date_parsed,occurrence_candidate_1,occurrence_candidate_2,lag_candidate_1,lag_candidate_2,candidate_1_valid,candidate_2_valid
4,CLM_0000005,08/03/2025,2025-04-12,2025-03-08,2025-08-03,35,-113,True,False
9,CLM_0000010,02/11/2025,2025-02-22,2025-11-02,2025-02-11,-253,11,False,True
74,CLM_0000075,10/08/2024,2024-08-29,2024-08-10,2024-10-08,19,-40,True,False
118,CLM_0000119,03/03/2025,2025-03-01,2025-03-03,2025-03-03,-2,-2,False,False
132,CLM_0000133,07/11/2025,2025-09-06,2025-11-07,2025-07-11,-62,57,False,True
154,CLM_0000155,10/12/2024,2024-09-23,2024-12-10,2024-10-12,-78,-19,False,False


### Ambiguous Occurrence Date Findings

Among the six claims with an ambiguous occurrence date and a resolved declaration date:

- Four claims have exactly one occurrence-date interpretation that produces a valid declaration lag between 0 and 62 days.
- These four records can therefore be resolved deterministically.
- Two claims have no valid occurrence-date interpretation under the current date ordering.

The two unresolved records show patterns consistent with the previously identified occurrence/declaration date reversal issue.

For one record, swapping the dates would produce a 2-day declaration delay.

For the second record, one ambiguous date interpretation combined with a date reversal would produce a 19-day declaration delay.

These two records will therefore be evaluated separately using the previously validated date-reversal rule before any correction is applied.

In [133]:
unresolved_occurrence = ambiguous_occurrence.loc[
    ~ambiguous_occurrence["candidate_1_valid"]
    & ~ambiguous_occurrence["candidate_2_valid"]
].copy()

unresolved_occurrence[
    [
        "claim_id",
        "occurrence_date",
        "declaration_date_parsed",
        "occurrence_candidate_1",
        "occurrence_candidate_2",
        "lag_candidate_1",
        "lag_candidate_2",
    ]
]

,claim_id,occurrence_date,declaration_date_parsed,occurrence_candidate_1,occurrence_candidate_2,lag_candidate_1,lag_candidate_2
118,CLM_0000119,03/03/2025,2025-03-01,2025-03-03,2025-03-03,-2,-2
154,CLM_0000155,10/12/2024,2024-09-23,2024-12-10,2024-10-12,-78,-19


In [134]:
unresolved_occurrence["swapped_lag_candidate_1"] = (
    unresolved_occurrence["occurrence_candidate_1"]
    - unresolved_occurrence["declaration_date_parsed"]
).dt.days

unresolved_occurrence["swapped_lag_candidate_2"] = (
    unresolved_occurrence["occurrence_candidate_2"]
    - unresolved_occurrence["declaration_date_parsed"]
).dt.days

In [135]:
unresolved_occurrence["swapped_candidate_1_valid"] = (
    unresolved_occurrence["swapped_lag_candidate_1"]
    .between(
        MIN_CLAIM_LAG_DAYS,
        MAX_CLAIM_LAG_DAYS,
    )
)

unresolved_occurrence["swapped_candidate_2_valid"] = (
    unresolved_occurrence["swapped_lag_candidate_2"]
    .between(
        MIN_CLAIM_LAG_DAYS,
        MAX_CLAIM_LAG_DAYS,
    )
)

unresolved_occurrence[
    [
        "claim_id",
        "occurrence_candidate_1",
        "occurrence_candidate_2",
        "swapped_lag_candidate_1",
        "swapped_lag_candidate_2",
        "swapped_candidate_1_valid",
        "swapped_candidate_2_valid",
    ]
]

,claim_id,occurrence_candidate_1,occurrence_candidate_2,swapped_lag_candidate_1,swapped_lag_candidate_2,swapped_candidate_1_valid,swapped_candidate_2_valid
118,CLM_0000119,2025-03-03,2025-03-03,2,2,True,True
154,CLM_0000155,2024-12-10,2024-10-12,78,19,False,True


### Ambiguous Occurrence Date Resolution

All six claims with an initially ambiguous occurrence date can be resolved using the validated declaration-lag rules.

Four records have exactly one direct date interpretation that produces a valid declaration delay between 0 and 62 days.

The remaining two records are consistent with the previously identified occurrence/declaration reversal issue:

- One record produces a valid 2-day declaration delay after swapping the dates.
- One record requires the second occurrence-date interpretation followed by a date swap, producing a valid 19-day declaration delay.

Therefore, all six ambiguous occurrence-date records can be resolved without leaving any unresolved values in this subgroup.

Records corrected through date reversal are also flagged in `claim_dates_swapped` for auditability.

In [136]:
direct_occurrence = ambiguous_occurrence.loc[
    ambiguous_occurrence["candidate_1_valid"]
    ^ ambiguous_occurrence["candidate_2_valid"]
].copy()

direct_occurrence["resolved_occurrence_date"] = np.where(
    direct_occurrence["candidate_1_valid"],
    direct_occurrence["occurrence_candidate_1"],
    direct_occurrence["occurrence_candidate_2"],
)

direct_occurrence[
    [
        "claim_id",
        "occurrence_date",
        "resolved_occurrence_date",
        "declaration_date_parsed",
    ]
]

,claim_id,occurrence_date,resolved_occurrence_date,declaration_date_parsed
4,CLM_0000005,08/03/2025,2025-03-08,2025-04-12
9,CLM_0000010,02/11/2025,2025-02-11,2025-02-22
74,CLM_0000075,10/08/2024,2024-08-10,2024-08-29
132,CLM_0000133,07/11/2025,2025-07-11,2025-09-06


In [137]:
direct_occurrence_map = (
    direct_occurrence
    .set_index("claim_id")["resolved_occurrence_date"]
)

mask = claims["claim_id"].isin(
    direct_occurrence_map.index
)

claims.loc[
    mask,
    "occurrence_date_parsed",
] = (
    claims.loc[mask, "claim_id"]
    .map(direct_occurrence_map)
)

In [138]:
mask_119 = claims["claim_id"].eq("CLM_0000119")

claims.loc[
    mask_119,
    "occurrence_date_parsed",
] = pd.Timestamp("2025-03-01")

claims.loc[
    mask_119,
    "declaration_date_parsed",
] = pd.Timestamp("2025-03-03")

claims.loc[
    mask_119,
    "claim_dates_swapped",
] = True

In [139]:
mask_155 = claims["claim_id"].eq("CLM_0000155")

claims.loc[
    mask_155,
    "occurrence_date_parsed",
] = pd.Timestamp("2024-09-23")

claims.loc[
    mask_155,
    "declaration_date_parsed",
] = pd.Timestamp("2024-10-12")

claims.loc[
    mask_155,
    "claim_dates_swapped",
] = True

In [140]:
occurrence_resolution_check = claims.loc[
    claims["claim_id"].isin(
        ambiguous_occurrence["claim_id"]
    ),
    [
        "claim_id",
        "occurrence_date",
        "declaration_date",
        "occurrence_date_parsed",
        "declaration_date_parsed",
        "claim_dates_swapped",
    ],
].copy()

occurrence_resolution_check["declaration_lag_days"] = (
    occurrence_resolution_check["declaration_date_parsed"]
    - occurrence_resolution_check["occurrence_date_parsed"]
).dt.days

occurrence_resolution_check

,claim_id,occurrence_date,declaration_date,occurrence_date_parsed,declaration_date_parsed,claim_dates_swapped,declaration_lag_days
4,CLM_0000005,08/03/2025,2025-04-12,2025-03-08,2025-04-12,False,35
9,CLM_0000010,02/11/2025,2025-02-22,2025-02-11,2025-02-22,False,11
74,CLM_0000075,10/08/2024,08/29/2024,2024-08-10,2024-08-29,False,19
118,CLM_0000119,03/03/2025,2025-03-01,2025-03-01,2025-03-03,True,2
132,CLM_0000133,07/11/2025,2025-09-06,2025-07-11,2025-09-06,False,57
154,CLM_0000155,10/12/2024,2024-09-23,2024-09-23,2024-10-12,True,19


In [141]:
print(
    "Remaining unresolved occurrence dates:",
    claims["occurrence_date_parsed"].isna().sum()
)

print(
    "Remaining unresolved declaration dates:",
    claims["declaration_date_parsed"].isna().sum()
)

print(
    "Negative lags among resolved claims:",
    (
        (
            claims["declaration_date_parsed"]
            - claims["occurrence_date_parsed"]
        ).dt.days < 0
    ).sum()
)

Remaining unresolved occurrence dates: 1
Remaining unresolved declaration dates: 9
Negative lags among resolved claims: 0


### Ambiguous Claim Declaration Dates

Claims with a resolved occurrence date but an ambiguous declaration date are evaluated using the validated declaration-delay range.

For each ambiguous declaration date, both possible day-month interpretations are generated.

A candidate declaration date is considered plausible when:

- It does not occur before the claim occurrence date.
- The resulting declaration lag falls within the empirically observed range of 0 to 62 days.

No date is modified until the candidate interpretations have been evaluated.

In [142]:
ambiguous_declaration = claims.loc[
    claims["declaration_date_parsed"].isna()
    & claims["occurrence_date_parsed"].notna(),
    [
        "claim_id",
        "occurrence_date_parsed",
        "declaration_date",
    ],
].copy()

print(
    "Claims with ambiguous declaration date:",
    len(ambiguous_declaration)
)

Claims with ambiguous declaration date: 8


In [143]:
declaration_candidates = (
    ambiguous_declaration["declaration_date"]
    .apply(get_ambiguous_date_candidates)
)

ambiguous_declaration[
    [
        "declaration_candidate_1",
        "declaration_candidate_2",
    ]
] = pd.DataFrame(
    declaration_candidates.tolist(),
    index=ambiguous_declaration.index,
)

In [144]:
ambiguous_declaration["lag_candidate_1"] = (
    ambiguous_declaration["declaration_candidate_1"]
    - ambiguous_declaration["occurrence_date_parsed"]
).dt.days

ambiguous_declaration["lag_candidate_2"] = (
    ambiguous_declaration["declaration_candidate_2"]
    - ambiguous_declaration["occurrence_date_parsed"]
).dt.days

In [145]:
ambiguous_declaration["candidate_1_valid"] = (
    ambiguous_declaration["lag_candidate_1"]
    .between(
        MIN_CLAIM_LAG_DAYS,
        MAX_CLAIM_LAG_DAYS,
    )
)

ambiguous_declaration["candidate_2_valid"] = (
    ambiguous_declaration["lag_candidate_2"]
    .between(
        MIN_CLAIM_LAG_DAYS,
        MAX_CLAIM_LAG_DAYS,
    )
)

In [146]:
declaration_candidate_summary = (
    ambiguous_declaration
    .groupby(
        [
            "candidate_1_valid",
            "candidate_2_valid",
        ]
    )
    .size()
    .rename("record_count")
    .reset_index()
)

declaration_candidate_summary

,candidate_1_valid,candidate_2_valid,record_count
0,False,False,4
1,False,True,2
2,True,False,1
3,True,True,1


In [147]:
ambiguous_declaration[
    [
        "claim_id",
        "occurrence_date_parsed",
        "declaration_date",
        "declaration_candidate_1",
        "declaration_candidate_2",
        "lag_candidate_1",
        "lag_candidate_2",
        "candidate_1_valid",
        "candidate_2_valid",
    ]
]

,claim_id,occurrence_date_parsed,declaration_date,declaration_candidate_1,declaration_candidate_2,lag_candidate_1,lag_candidate_2,candidate_1_valid,candidate_2_valid
13,CLM_0000014,2025-05-31,06/07/2025,2025-07-06,2025-06-07,36,7,True,True
15,CLM_0000016,2025-05-12,05/12/2025,2025-12-05,2025-05-12,207,0,False,True
19,CLM_0000020,2023-06-26,01/07/2023,2023-07-01,2023-01-07,5,-170,True,False
31,CLM_0000032,2025-11-19,11/08/2025,2025-08-11,2025-11-08,-100,-11,False,False
106,CLM_0000107,2025-12-18,05/12/2025,2025-12-05,2025-05-12,-13,-220,False,False
116,CLM_0000117,2025-05-11,05/05/2025,2025-05-05,2025-05-05,-6,-6,False,False
141,CLM_0000142,2025-08-31,10/12/2025,2025-12-10,2025-10-12,101,42,False,True
144,CLM_0000145,2024-01-31,12/01/2024,2024-01-12,2024-12-01,-19,305,False,False


### Ambiguous Declaration Date Findings

Among the eight claims with an ambiguous declaration date and a resolved occurrence date:

- Three claims have exactly one declaration-date interpretation that produces a valid declaration lag between 0 and 62 days.
- One claim has two plausible declaration-date interpretations, producing delays of 36 and 7 days respectively.
- Four claims have no valid declaration-date interpretation under the current date ordering.

The four records with no valid direct interpretation show patterns consistent with the previously identified occurrence/declaration date reversal issue.

The record with two valid interpretations will remain unresolved until sufficient evidence is available to distinguish between the two plausible dates.

In [148]:
unresolved_declaration = ambiguous_declaration.loc[
    ~ambiguous_declaration["candidate_1_valid"]
    & ~ambiguous_declaration["candidate_2_valid"]
].copy()

unresolved_declaration[
    [
        "claim_id",
        "occurrence_date_parsed",
        "declaration_date",
        "declaration_candidate_1",
        "declaration_candidate_2",
        "lag_candidate_1",
        "lag_candidate_2",
    ]
]

,claim_id,occurrence_date_parsed,declaration_date,declaration_candidate_1,declaration_candidate_2,lag_candidate_1,lag_candidate_2
31,CLM_0000032,2025-11-19,11/08/2025,2025-08-11,2025-11-08,-100,-11
106,CLM_0000107,2025-12-18,05/12/2025,2025-12-05,2025-05-12,-13,-220
116,CLM_0000117,2025-05-11,05/05/2025,2025-05-05,2025-05-05,-6,-6
144,CLM_0000145,2024-01-31,12/01/2024,2024-01-12,2024-12-01,-19,305


In [149]:
unresolved_declaration["swapped_lag_candidate_1"] = (
    unresolved_declaration["occurrence_date_parsed"]
    - unresolved_declaration["declaration_candidate_1"]
).dt.days

unresolved_declaration["swapped_lag_candidate_2"] = (
    unresolved_declaration["occurrence_date_parsed"]
    - unresolved_declaration["declaration_candidate_2"]
).dt.days

In [150]:
unresolved_declaration["swapped_candidate_1_valid"] = (
    unresolved_declaration["swapped_lag_candidate_1"]
    .between(
        MIN_CLAIM_LAG_DAYS,
        MAX_CLAIM_LAG_DAYS,
    )
)

unresolved_declaration["swapped_candidate_2_valid"] = (
    unresolved_declaration["swapped_lag_candidate_2"]
    .between(
        MIN_CLAIM_LAG_DAYS,
        MAX_CLAIM_LAG_DAYS,
    )
)

unresolved_declaration[
    [
        "claim_id",
        "declaration_candidate_1",
        "declaration_candidate_2",
        "swapped_lag_candidate_1",
        "swapped_lag_candidate_2",
        "swapped_candidate_1_valid",
        "swapped_candidate_2_valid",
    ]
]

,claim_id,declaration_candidate_1,declaration_candidate_2,swapped_lag_candidate_1,swapped_lag_candidate_2,swapped_candidate_1_valid,swapped_candidate_2_valid
31,CLM_0000032,2025-08-11,2025-11-08,100,11,False,True
106,CLM_0000107,2025-12-05,2025-05-12,13,220,True,False
116,CLM_0000117,2025-05-05,2025-05-05,6,6,True,True
144,CLM_0000145,2024-01-12,2024-12-01,19,-305,True,False


### Declaration Date Reversal Validation

The four claims without a valid direct declaration-date interpretation were evaluated under the previously validated date-reversal hypothesis.

After reversing the chronological roles of the dates:

- `CLM_0000032` produces an 11-day declaration lag.
- `CLM_0000107` produces a 13-day declaration lag.
- `CLM_0000117` produces a 6-day declaration lag.
- `CLM_0000145` produces a 19-day declaration lag.

All corrected delays fall within the empirically validated 0–62 day declaration-lag range.

Therefore, these four records are treated as additional occurrence/declaration reversal cases and can be corrected consistently with the previously established cleaning rule.

In [151]:
direct_declaration = ambiguous_declaration.loc[
    ambiguous_declaration["candidate_1_valid"]
    ^ ambiguous_declaration["candidate_2_valid"]
].copy()

direct_declaration["resolved_declaration_date"] = np.where(
    direct_declaration["candidate_1_valid"],
    direct_declaration["declaration_candidate_1"],
    direct_declaration["declaration_candidate_2"],
)

direct_declaration[
    [
        "claim_id",
        "occurrence_date_parsed",
        "resolved_declaration_date",
    ]
]

,claim_id,occurrence_date_parsed,resolved_declaration_date
15,CLM_0000016,2025-05-12,2025-05-12
19,CLM_0000020,2023-06-26,2023-07-01
141,CLM_0000142,2025-08-31,2025-10-12


In [152]:
direct_declaration_map = (
    direct_declaration
    .set_index("claim_id")["resolved_declaration_date"]
)

mask = claims["claim_id"].isin(
    direct_declaration_map.index
)

claims.loc[
    mask,
    "declaration_date_parsed",
] = (
    claims.loc[mask, "claim_id"]
    .map(direct_declaration_map)
)

In [153]:
reversed_declaration_rules = {
    "CLM_0000032": pd.Timestamp("2025-11-08"),
    "CLM_0000107": pd.Timestamp("2025-12-05"),
    "CLM_0000117": pd.Timestamp("2025-05-05"),
    "CLM_0000145": pd.Timestamp("2024-01-12"),
}

In [154]:
for claim_id, corrected_occurrence in reversed_declaration_rules.items():

    mask = claims["claim_id"].eq(claim_id)

    previous_occurrence = claims.loc[
        mask,
        "occurrence_date_parsed"
    ].iloc[0]

    claims.loc[
        mask,
        "occurrence_date_parsed"
    ] = corrected_occurrence

    claims.loc[
        mask,
        "declaration_date_parsed"
    ] = previous_occurrence

    claims.loc[
        mask,
        "claim_dates_swapped"
    ] = True

In [155]:
claim_lag = (
    claims["declaration_date_parsed"]
    - claims["occurrence_date_parsed"]
).dt.days

print(
    "Remaining unresolved occurrence dates:",
    claims["occurrence_date_parsed"].isna().sum()
)

print(
    "Remaining unresolved declaration dates:",
    claims["declaration_date_parsed"].isna().sum()
)

print(
    "Negative lags among resolved claims:",
    (claim_lag < 0).sum()
)

print(
    "Lags above 62 days:",
    (claim_lag > 62).sum()
)

Remaining unresolved occurrence dates: 1
Remaining unresolved declaration dates: 2
Negative lags among resolved claims: 0
Lags above 62 days: 0


### Remaining Unresolved Claim Dates

After applying explicit parsing rules, declaration-lag constraints, and validated occurrence/declaration reversal corrections, only two claim records still contain unresolved date information.

These remaining records are inspected individually before any final cleaning decision is made.

Dates will only be resolved when the available evidence provides a sufficiently reliable interpretation. Otherwise, the ambiguous values will remain unresolved to avoid introducing unsupported assumptions.

In [156]:
remaining_claim_date_issues = claims.loc[
    claims["occurrence_date_parsed"].isna()
    | claims["declaration_date_parsed"].isna(),
    [
        "claim_id",
        "contract_id",
        "occurrence_date",
        "declaration_date",
        "occurrence_date_parsed",
        "declaration_date_parsed",
        "claim_dates_swapped",
        "status",
    ],
].copy()

remaining_claim_date_issues

,claim_id,contract_id,occurrence_date,declaration_date,occurrence_date_parsed,declaration_date_parsed,claim_dates_swapped,status
13,CLM_0000014,CTR_007351,2025-05-31,06/07/2025,2025-05-31,NaT,False,Open
143,CLM_0000144,CTR_002798,10/07/2025,10/01/2025,NaT,NaT,False,Expert_review


In [157]:
for _, row in remaining_claim_date_issues.iterrows():

    print("=" * 70)
    print("Claim ID:", row["claim_id"])

    occurrence_candidates = get_ambiguous_date_candidates(
        row["occurrence_date"]
    )

    declaration_candidates = get_ambiguous_date_candidates(
        row["declaration_date"]
    )

    print("Raw occurrence :", row["occurrence_date"])
    print("Occurrence candidates:", occurrence_candidates)

    print("Raw declaration:", row["declaration_date"])
    print("Declaration candidates:", declaration_candidates)

    print(
        "Current parsed occurrence:",
        row["occurrence_date_parsed"]
    )

    print(
        "Current parsed declaration:",
        row["declaration_date_parsed"]
    )

Claim ID: CLM_0000014
Raw occurrence : 2025-05-31
Occurrence candidates: (NaT, NaT)
Raw declaration: 06/07/2025
Declaration candidates: (Timestamp('2025-07-06 00:00:00'), Timestamp('2025-06-07 00:00:00'))
Current parsed occurrence: 2025-05-31 00:00:00
Current parsed declaration: NaT
Claim ID: CLM_0000144
Raw occurrence : 10/07/2025
Occurrence candidates: (Timestamp('2025-07-10 00:00:00'), Timestamp('2025-10-07 00:00:00'))
Raw declaration: 10/01/2025
Declaration candidates: (Timestamp('2025-01-10 00:00:00'), Timestamp('2025-10-01 00:00:00'))
Current parsed occurrence: NaT
Current parsed declaration: NaT


### Final Unresolved Claim Date Assessment

Two claim records remain after the previous date-cleaning steps.

For `CLM_0000014`, the occurrence date is known, but both possible declaration-date interpretations produce plausible declaration delays. Since the available data does not provide sufficient evidence to distinguish between these alternatives, the declaration date will remain unresolved.

`CLM_0000144` contains ambiguous values in both occurrence and declaration date fields. All possible date combinations, including the previously validated occurrence/declaration reversal scenario, are evaluated before making a final cleaning decision.

In [158]:
claim_144 = claims.loc[
    claims["claim_id"].eq("CLM_0000144")
].iloc[0]

occ_candidates = get_ambiguous_date_candidates(
    claim_144["occurrence_date"]
)

dec_candidates = get_ambiguous_date_candidates(
    claim_144["declaration_date"]
)

candidate_rows = []

for occ_candidate in occ_candidates:
    for dec_candidate in dec_candidates:

        direct_lag = (
            dec_candidate - occ_candidate
        ).days

        swapped_lag = (
            occ_candidate - dec_candidate
        ).days

        candidate_rows.append(
            {
                "occurrence_candidate": occ_candidate,
                "declaration_candidate": dec_candidate,
                "direct_lag": direct_lag,
                "direct_valid": (
                    MIN_CLAIM_LAG_DAYS
                    <= direct_lag
                    <= MAX_CLAIM_LAG_DAYS
                ),
                "swapped_lag": swapped_lag,
                "swapped_valid": (
                    MIN_CLAIM_LAG_DAYS
                    <= swapped_lag
                    <= MAX_CLAIM_LAG_DAYS
                ),
            }
        )

claim_144_candidates = pd.DataFrame(candidate_rows)

claim_144_candidates

,occurrence_candidate,declaration_candidate,direct_lag,direct_valid,swapped_lag,swapped_valid
0,2025-07-10,2025-01-10,-181,False,181,False
1,2025-07-10,2025-10-01,83,False,-83,False
2,2025-10-07,2025-01-10,-270,False,270,False
3,2025-10-07,2025-10-01,-6,False,6,True


### Final Claim Date Resolution

The final unresolved claim with both dates ambiguous (`CLM_0000144`) was evaluated across all possible occurrence and declaration date combinations.

Only one interpretation produces a valid declaration delay when the previously validated date-reversal rule is applied:

- Corrected occurrence date: `2025-10-01`
- Corrected declaration date: `2025-10-07`
- Declaration delay: 6 days

Therefore, this record can be resolved consistently with the date-reversal pattern identified in other claim records.

One claim (`CLM_0000014`) remains partially unresolved because two different declaration-date interpretations both produce plausible declaration delays of 7 and 36 days.

Since no available evidence reliably distinguishes between these alternatives, its cleaned declaration date is intentionally preserved as `NaT`.

In [159]:
mask_144 = claims["claim_id"].eq("CLM_0000144")

claims.loc[
    mask_144,
    "occurrence_date_parsed"
] = pd.Timestamp("2025-10-01")

claims.loc[
    mask_144,
    "declaration_date_parsed"
] = pd.Timestamp("2025-10-07")

claims.loc[
    mask_144,
    "claim_dates_swapped"
] = True

In [160]:
claims["declaration_lag_days"] = (
    claims["declaration_date_parsed"]
    - claims["occurrence_date_parsed"]
).dt.days

In [161]:
print(
    "Resolved occurrence dates:",
    claims["occurrence_date_parsed"].notna().sum()
)

print(
    "Unresolved occurrence dates:",
    claims["occurrence_date_parsed"].isna().sum()
)

print()

print(
    "Resolved declaration dates:",
    claims["declaration_date_parsed"].notna().sum()
)

print(
    "Unresolved declaration dates:",
    claims["declaration_date_parsed"].isna().sum()
)

print()

print(
    "Negative declaration lags:",
    (claims["declaration_lag_days"] < 0).sum()
)

print(
    "Declaration lags above 62 days:",
    (claims["declaration_lag_days"] > 62).sum()
)

print()

print(
    "Claims corrected by date reversal:",
    claims["claim_dates_swapped"].sum()
)

Resolved occurrence dates: 155
Unresolved occurrence dates: 0

Resolved declaration dates: 154
Unresolved declaration dates: 1

Negative declaration lags: 0
Declaration lags above 62 days: 0

Claims corrected by date reversal: 64


In [162]:
claims.loc[
    claims["declaration_date_parsed"].isna(),
    [
        "claim_id",
        "occurrence_date",
        "declaration_date",
        "occurrence_date_parsed",
        "declaration_date_parsed",
        "status",
    ],
]

,claim_id,occurrence_date,declaration_date,occurrence_date_parsed,declaration_date_parsed,status
13,CLM_0000014,2025-05-31,06/07/2025,2025-05-31,NaT,Open


### Date Standardization Summary

The contract and claim date-cleaning process successfully standardized the majority of mixed-format date values while avoiding unsupported assumptions.

For contract dates:

- 14,996 of 15,000 start dates were resolved.
- 14,995 of 15,000 end dates were resolved.
- Only 7 contracts retain at least one unresolved date because multiple interpretations remain equally plausible.

For claim dates:

- All 155 occurrence dates were resolved.
- 154 of 155 declaration dates were resolved.
- 64 claim records were corrected using a validated occurrence/declaration date-reversal rule.
- No resolved claim contains a negative declaration lag.
- No resolved declaration lag exceeds the empirically observed 62-day maximum.
- One declaration date (`CLM_0000014`) remains unresolved because two alternative interpretations are equally plausible.

The unresolved values are intentionally preserved as missing rather than being assigned arbitrarily.

The cleaned date fields now provide a consistent chronological foundation for SQL analysis, Power BI reporting, and future feature engineering.

## 5. Categorical and Structural Standardization

Categorical variables are standardized to ensure consistent representations across the cleaned datasets.

The data-understanding stage identified only a small number of categorical inconsistencies.

The first issue addressed is the `gender` variable, where the same categories are represented using both abbreviated and full labels:

- `F` and `Female`
- `M` and `Male`

These representations will be consolidated while preserving existing missing values.

## 5. Categorical and Structural Standardization

Categorical variables are standardized to ensure consistent representations across the cleaned datasets.

The data-understanding stage identified only a small number of categorical inconsistencies.

The first issue addressed is the `gender` variable, where the same categories are represented using both abbreviated and full labels:

- `F` and `Female`
- `M` and `Male`

These representations will be consolidated while preserving existing missing values.

In [164]:
gender_before = (
    contracts["gender"]
    .value_counts(dropna=False)
    .rename_axis("gender")
    .reset_index(name="count")
)

gender_before

,gender,count
0,NaN,3096
1,Male,3029
2,Female,2993
3,M,2949
4,F,2933


In [165]:
gender_mapping = {
    "F": "Female",
    "M": "Male",
}

contracts["gender"] = (
    contracts["gender"]
    .replace(gender_mapping)
)

In [166]:
gender_after = (
    contracts["gender"]
    .value_counts(dropna=False)
    .rename_axis("gender")
    .reset_index(name="count")
)

gender_after

,gender,count
0,Male,5978
1,Female,5926
2,NaN,3096


In [167]:
print(
    "Unique gender values:",
    contracts["gender"].dropna().unique()
)

print(
    "Missing values before:",
    contracts_raw["gender"].isna().sum()
)

print(
    "Missing values after:",
    contracts["gender"].isna().sum()
)

Unique gender values: <StringArray>
['Female', 'Male']
Length: 2, dtype: str
Missing values before: 3096
Missing values after: 3096


### Gender Standardization Findings

The `gender` variable was successfully standardized by consolidating abbreviated and full category labels.

The following mappings were applied:

- `F` → `Female`
- `M` → `Male`

After standardization, only two non-missing categories remain:

- `Female`
- `Male`

The original 3,096 missing values were preserved and no additional missing observations were introduced.

This transformation improves categorical consistency for SQL analysis, Power BI reporting, and future machine learning preprocessing.

### City and Postal Code Separation

The `city_postal` variable combines two distinct attributes within a single field.

Examples include:

- `Paris_75001`
- `Lyon_69000`
- `Bordeaux_33000`

For a cleaner relational structure and more flexible geographic analysis, this field is separated into:

- `city`
- `postal_code`

The original `city_postal` column is retained temporarily for validation.

In [168]:
city_postal_split = (
    contracts["city_postal"]
    .astype("string")
    .str.rsplit("_", n=1, expand=True)
)

contracts["city"] = city_postal_split[0]
contracts["postal_code"] = city_postal_split[1]

In [169]:
contracts[
    [
        "city_postal",
        "city",
        "postal_code",
    ]
].drop_duplicates().sort_values("city")

,city_postal,city,postal_code
1,Bordeaux_33000,Bordeaux,33000
8,Dijon_21000,Dijon,21000
7,Lyon_69000,Lyon,69000
4,Marseille_13000,Marseille,13000
22,Nantes_44000,Nantes,44000
0,Paris_75001,Paris,75001
14,Toulouse_31000,Toulouse,31000


In [170]:
print(
    "Missing city values:",
    contracts["city"].isna().sum()
)

print(
    "Missing postal code values:",
    contracts["postal_code"].isna().sum()
)

print(
    "Unique cities:",
    contracts["city"].nunique()
)

print(
    "Unique postal codes:",
    contracts["postal_code"].nunique()
)

Missing city values: 0
Missing postal code values: 0
Unique cities: 7
Unique postal codes: 7


### City and Postal Code Separation Findings

The `city_postal` variable was successfully separated into two independent attributes:

- `city`
- `postal_code`

All seven original city-postal combinations were parsed successfully.

No missing values were introduced during the transformation, and the resulting dataset contains seven unique cities and seven corresponding postal codes.

The original `city_postal` field is retained temporarily for validation and will be evaluated for removal when the final cleaned schema is defined.

## 6. Vehicle Power Standardization

The `power` variable contains several representations of vehicle engine power:

- Horsepower (`HP`)
- Metric horsepower (`CV`)
- Kilowatts (`kW`)
- Numeric values without an explicit unit
- Missing values

The data-understanding stage showed that explicitly labeled HP, CV, and kW observations represent highly similar underlying power distributions after conversion to horsepower.

Plain numeric values also closely match the HP/CV distribution.

The cleaning strategy therefore:

- Extracts the numeric component of each power value.
- Identifies the original measurement unit where available.
- Converts explicitly labeled CV and kW values to horsepower.
- Treats unitless numeric values as HP-equivalent under a documented assumption.
- Preserves missing values.
- Retains unit information for auditability.

In [171]:
power_str = (
    vehicles["power"]
    .astype("string")
    .str.strip()
)

vehicles["power_value_raw"] = (
    power_str
    .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
    .astype("Float64")
)

In [172]:
vehicles["power_unit"] = np.select(
    [
        power_str.str.contains(
            "hp",
            case=False,
            regex=False,
            na=False,
        ),
        power_str.str.contains(
            "cv",
            case=False,
            regex=False,
            na=False,
        ),
        power_str.str.contains(
            "kw",
            case=False,
            regex=False,
            na=False,
        ),
        power_str.str.fullmatch(
            r"\d+(?:\.\d+)?",
            na=False,
        ),
    ],
    [
        "HP",
        "CV",
        "kW",
        "Unitless",
    ],
    default="Unknown",
)

In [173]:
vehicles["power_unit"] = (
    vehicles["power_unit"]
    .astype("string")
    .mask(
        vehicles["power"].isna(),
        pd.NA,
    )
)

In [175]:
vehicles["power_hp"] = pd.Series(
    pd.NA,
    index=vehicles.index,
    dtype="Float64",
)

vehicles.loc[
    vehicles["power_unit"].eq("HP").fillna(False),
    "power_hp",
] = vehicles["power_value_raw"]

vehicles.loc[
    vehicles["power_unit"].eq("CV").fillna(False),
    "power_hp",
] = (
    vehicles["power_value_raw"] * 0.98632
)

vehicles.loc[
    vehicles["power_unit"].eq("kW").fillna(False),
    "power_hp",
] = (
    vehicles["power_value_raw"] * 1.34102
)

vehicles.loc[
    vehicles["power_unit"].eq("Unitless").fillna(False),
    "power_hp",
] = vehicles["power_value_raw"]

vehicles["power_hp"] = vehicles["power_hp"].round(2)

In [176]:
vehicles["power_unit"].value_counts(dropna=False)

power_unit
HP          1797
Unitless     904
<NA>         900
kW           895
CV           894
Name: count, dtype: Int64

In [177]:
print(
    "Original missing power values:",
    vehicles_raw["power"].isna().sum()
)

print(
    "Missing standardized power values:",
    vehicles["power_hp"].isna().sum()
)

print(
    "Unexpected power units:",
    vehicles["power_unit"].eq("Unknown").fillna(False).sum()
)

Original missing power values: 900
Missing standardized power values: 900
Unexpected power units: 0


In [178]:
vehicles[
    [
        "power",
        "power_value_raw",
        "power_unit",
        "power_hp",
    ]
].sample(
    20,
    random_state=42,
)

,power,power_value_raw,power_unit,power_hp
2810,NaN,<NA>,<NA>,<NA>
2316,114 kW,114.00,kW,152.88
5086,143 CV,143.00,CV,141.04
3084,98hp,98.00,HP,98.00
3039,83 kW,83.00,kW,111.30
3282,194hp,194.00,HP,194.00
964,75 kW,75.00,kW,100.58
3159,130 HP,130.00,HP,130.00
491,199,199.00,Unitless,199.00
4563,NaN,<NA>,<NA>,<NA>


### Vehicle Power Standardization Findings

Vehicle power values were successfully standardized into a common horsepower representation.

The original power formats were classified as:

- HP: 1,797 records
- CV: 894 records
- kW: 895 records
- Unitless numeric values: 904 records
- Missing: 900 records

No unexpected power formats remained after classification.

Explicitly labeled CV and kW values were converted to horsepower using documented conversion factors, while HP values were preserved directly.

Unitless numeric values were treated as HP-equivalent based on the distributional evidence identified during the data-understanding stage. Because the original unit cannot be verified with certainty, the `power_unit` field is retained to document this assumption.

All 900 original missing power values remained missing after standardization, and no additional missing observations were introduced.

## 7. Missing Value Treatment Strategy

Missing values are handled according to their business meaning rather than being filled automatically.

The cleaning strategy distinguishes between three types of missingness:

### Structural or Business Missingness

Some missing values represent a legitimate business state rather than incomplete data.

For example, `indemnified_amount` is unavailable for claims that have not yet reached a finalized settlement stage.

These values are preserved as missing.

### Variables Requiring Statistical Imputation

Variables such as customer age or vehicle characteristics may eventually require imputation for machine learning.

However, statistical imputation is not performed during the general data-cleaning stage because calculating imputation values from the complete dataset could introduce data leakage.

These values will be handled later within machine-learning preprocessing pipelines using training data only.

### Deterministically Recoverable Values

Missing values will only be filled during data cleaning when they can be reconstructed from available information using a reliable business or logical rule.

This approach preserves data integrity while keeping the cleaned datasets suitable for SQL, Power BI, and future machine-learning workflows.

In [179]:
missing_columns = {
    "contracts": [
        "client_age",
        "csp",
        "gender",
    ],
    "claims": [
        "indemnified_amount",
        "expert_id",
        "liability",
    ],
    "vehicles": [
        "year",
        "power",
        "color",
        "previous_claims",
    ],
}

missing_summary_clean = []

for dataset_name, columns in missing_columns.items():
    df = datasets[dataset_name]

    for column in columns:
        missing_summary_clean.append(
            {
                "dataset": dataset_name,
                "column": column,
                "missing_count": df[column].isna().sum(),
                "missing_pct": df[column].isna().mean() * 100,
            }
        )

missing_summary_clean = (
    pd.DataFrame(missing_summary_clean)
    .sort_values(
        ["dataset", "missing_pct"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

missing_summary_clean

,dataset,column,missing_count,missing_pct
0,claims,indemnified_amount,65,41.94
1,claims,liability,47,30.32
2,claims,expert_id,38,24.52
3,contracts,gender,3096,20.64
4,contracts,csp,1772,11.81
5,contracts,client_age,1265,8.43
6,vehicles,power,900,16.70
7,vehicles,color,861,15.97
8,vehicles,previous_claims,535,9.93
9,vehicles,year,273,5.06


In [180]:
for dataset_name, columns in missing_columns.items():

    clean_df = datasets[dataset_name]

    raw_df = {
        "contracts": contracts_raw,
        "claims": claims_raw,
        "vehicles": vehicles_raw,
    }[dataset_name]

    print(f"\n{dataset_name.upper()}")
    print("-" * 50)

    for column in columns:

        raw_missing = raw_df[column].isna().sum()
        clean_missing = clean_df[column].isna().sum()

        print(
            f"{column:<20} "
            f"raw={raw_missing:<5} "
            f"clean={clean_missing:<5} "
            f"difference={clean_missing - raw_missing}"
        )


CONTRACTS
--------------------------------------------------
client_age           raw=1265  clean=1265  difference=0
csp                  raw=1772  clean=1772  difference=0
gender               raw=3096  clean=3096  difference=0

CLAIMS
--------------------------------------------------
indemnified_amount   raw=65    clean=65    difference=0
expert_id            raw=38    clean=38    difference=0
liability            raw=47    clean=47    difference=0

VEHICLES
--------------------------------------------------
year                 raw=273   clean=273   difference=0
power                raw=900   clean=900   difference=0
color                raw=861   clean=861   difference=0
previous_claims      raw=535   clean=535   difference=0


### Contract Missing Value Treatment

Missing customer attributes in the contracts dataset are intentionally preserved at this stage.

The affected variables are:

- `gender`: 3,096 missing observations (20.64%)
- `csp`: 1,772 missing observations (11.81%)
- `client_age`: 1,265 missing observations (8.43%)

No deterministic information is available to reconstruct these values reliably.

Therefore:

- Missing `client_age` values are preserved as null and will be handled later within the machine-learning preprocessing pipeline using training data only.
- Missing `csp` values are preserved rather than being replaced with the most frequent category.
- Missing `gender` values are preserved rather than inferred or assigned arbitrarily.

This avoids introducing artificial customer characteristics and prevents potential data leakage from dataset-wide statistical imputation.

For reporting purposes, missing categorical values may later be displayed as `Unknown` within the Power BI semantic layer without modifying the underlying cleaned data.

In [181]:
contract_missing_validation = pd.DataFrame(
    {
        "column": [
            "client_age",
            "csp",
            "gender",
        ],
        "missing_count": [
            contracts["client_age"].isna().sum(),
            contracts["csp"].isna().sum(),
            contracts["gender"].isna().sum(),
        ],
        "non_missing_count": [
            contracts["client_age"].notna().sum(),
            contracts["csp"].notna().sum(),
            contracts["gender"].notna().sum(),
        ],
        "unique_non_missing": [
            contracts["client_age"].nunique(dropna=True),
            contracts["csp"].nunique(dropna=True),
            contracts["gender"].nunique(dropna=True),
        ],
    }
)

contract_missing_validation

,column,missing_count,non_missing_count,unique_non_missing
0,client_age,1265,13735,58
1,csp,1772,13228,7
2,gender,3096,11904,2


In [182]:
contracts["client_age"].describe()

count   13,735.00
mean        44.44
std         11.77
min         18.00
25%         36.00
50%         44.00
75%         53.00
max         75.00
Name: client_age, dtype: float64

### Contract Missing Value Validation Findings

The remaining non-missing contract attributes were validated after categorical standardization.

- `client_age` contains valid observations between 18 and 75 years.
- `csp` contains seven distinct non-missing categories.
- `gender` contains two standardized categories: `Female` and `Male`.
- No additional missing values were introduced during cleaning.

Since the missing values cannot be reconstructed deterministically, they are preserved as null values in the cleaned dataset.

Statistical imputation, where required, will be performed later within machine-learning preprocessing pipelines using training data only.

### Claim Missing Value Treatment

Missing values in claim-related attributes are evaluated according to their operational meaning.

Previous analysis showed that missing `indemnified_amount` values are strongly associated with claims that have not yet reached a finalized settlement stage.

Therefore, claim-related missing values are not automatically treated as ordinary data-quality errors.

In [183]:
indemnified_by_status = (
    claims
    .groupby("status")
    .agg(
        total_claims=("claim_id", "count"),
        missing_indemnified=(
            "indemnified_amount",
            lambda x: x.isna().sum()
        ),
        zero_indemnified=(
            "indemnified_amount",
            lambda x: x.eq(0).sum()
        ),
        positive_indemnified=(
            "indemnified_amount",
            lambda x: x.gt(0).sum()
        ),
    )
)

indemnified_by_status["missing_pct"] = (
    indemnified_by_status["missing_indemnified"]
    / indemnified_by_status["total_claims"]
    * 100
)

indemnified_by_status

,total_claims,missing_indemnified,zero_indemnified,positive_indemnified,missing_pct
status,,,,,
Closed,47,0,6,41,0.00
Expert_review,25,25,0,0,100.00
In_progress,23,23,0,0,100.00
Open,17,17,0,0,100.00
Rejected,43,0,43,0,0.00


### Indemnified Amount Treatment Findings

The relationship between claim status and `indemnified_amount` confirms that missing and zero values represent different business states.

The observed pattern is:

- All Rejected claims contain an explicit indemnified amount of zero.
- Closed claims contain finalized indemnification values, which may be either zero or positive.
- All Open, In Progress, and Expert Review claims contain missing indemnification values.

Therefore:

- A zero indemnified amount represents a finalized claim with no indemnity payment.
- A missing indemnified amount represents a claim whose settlement amount has not yet been finalized.

Missing `indemnified_amount` values are therefore preserved as null values and are not replaced with zero.

This distinction is important for claims reporting, claim severity modeling, and future expected-loss calculations.

### Expert Identifier Validation

The `expert_id` variable contains missing values across multiple claim statuses.

Since `expert_id` represents an operational identifier rather than a continuous or ordinal attribute, missing values cannot be meaningfully imputed using statistical methods.

Before preserving these values as null, the available expert identifiers are validated for formatting consistency and uniqueness.

In [184]:
expert_summary = pd.Series(
    {
        "total_claims": len(claims),
        "missing_expert_id": claims["expert_id"].isna().sum(),
        "available_expert_id": claims["expert_id"].notna().sum(),
        "unique_experts": claims["expert_id"].nunique(dropna=True),
    }
)

expert_summary

total_claims           155
missing_expert_id       38
available_expert_id    117
unique_experts          28
dtype: int64

In [185]:
claims["expert_id"].dropna().sort_values().unique()

<StringArray>
['EXP_001', 'EXP_002', 'EXP_003', 'EXP_004', 'EXP_005', 'EXP_006', 'EXP_007',
 'EXP_008', 'EXP_009', 'EXP_011', 'EXP_012', 'EXP_013', 'EXP_014', 'EXP_015',
 'EXP_016', 'EXP_017', 'EXP_018', 'EXP_019', 'EXP_020', 'EXP_024', 'EXP_025',
 'EXP_030', 'EXP_033', 'EXP_034', 'EXP_035', 'EXP_036', 'EXP_039', 'EXP_041']
Length: 28, dtype: str

### Expert Identifier Treatment Findings

The available `expert_id` values follow a consistent operational identifier format such as `EXP_001`, `EXP_002`, and `EXP_003`.

Key observations include:

- 117 claims contain an expert identifier.
- 38 claims have no recorded expert identifier.
- 28 distinct experts are represented in the dataset.
- No inconsistent expert identifier formats were detected.

Since `expert_id` is an operational identifier rather than a measurable customer or risk attribute, missing expert identifiers are preserved as null values and are not imputed.

The variable may be useful for operational claims analysis and workload reporting. However, it should not be used as a predictor in a claim-occurrence model because expert assignment occurs after a claim has already been created and would therefore introduce data leakage.

### Liability Validation

The `liability` variable describes the responsibility classification associated with a claim.

Missing liability values are observed across several claim statuses and do not follow a deterministic lifecycle rule.

Before deciding how these missing values should be handled, the distribution of available liability categories is examined.

In [186]:
liability_summary = (
    claims["liability"]
    .value_counts(dropna=False)
    .rename_axis("liability")
    .reset_index(name="claim_count")
)

liability_summary["claim_pct"] = (
    liability_summary["claim_count"]
    / len(claims)
    * 100
)

liability_summary

,liability,claim_count,claim_pct
0,NaN,47,30.32
1,Insured,41,26.45
2,Third_party,33,21.29
3,Shared,29,18.71
4,Force_majeure,5,3.23


In [187]:
liability_by_status = pd.crosstab(
    claims["status"],
    claims["liability"],
    dropna=False,
)

liability_by_status

liability,Force_majeure,Insured,Shared,Third_party,NaN
status,,,,,
Closed,1,9,13,10,14
Expert_review,1,9,2,8,5
In_progress,3,9,3,2,6
Open,0,4,5,5,3
Rejected,0,10,6,8,19


### Liability Missing Value Treatment Findings

The `liability` variable contains four observed responsibility categories:

- `Insured`
- `Third_party`
- `Shared`
- `Force_majeure`

A total of 47 claims (30.32%) have no recorded liability classification.

Missing liability values are present across all claim statuses and do not follow a deterministic claim-lifecycle pattern.

Since the missing values cannot be reconstructed reliably from the available information, they are preserved as null values in the cleaned dataset.

An artificial `Unknown` category is not introduced at the data-cleaning stage because it could be interpreted as a genuine liability class.

If required:

- Power BI may display null values using a reporting label such as `Unknown / Not Available`.
- Machine-learning pipelines may handle missing liability values using an explicit missing category or another training-only preprocessing strategy.

No imputation is performed in the cleaned source data.

### Previous Claims Missing Value Assessment

The `previous_claims` variable represents the historical number of claims associated with a vehicle.

Before treating missing values as zero, the observed distribution is examined.

A missing value should not automatically be interpreted as "no previous claims" unless the data provides evidence that this is the intended business representation.

In [188]:
previous_claims_summary = (
    vehicles["previous_claims"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("previous_claims")
    .reset_index(name="vehicle_count")
)

previous_claims_summary["vehicle_pct"] = (
    previous_claims_summary["vehicle_count"]
    / len(vehicles)
    * 100
)

previous_claims_summary

,previous_claims,vehicle_count,vehicle_pct
0,0.00,1214,22.52
1,1.00,1245,23.10
2,2.00,1166,21.63
3,3.00,1230,22.82
4,NaN,535,9.93


In [189]:
vehicles["previous_claims"].describe()

count   4,855.00
mean        1.50
std         1.12
min         0.00
25%         0.50
50%         1.00
75%         3.00
max         3.00
Name: previous_claims, dtype: float64

### Previous Claims Missing Value Findings

The `previous_claims` variable contains explicit values ranging from 0 to 3.

The observed distribution is relatively balanced across the four recorded values, while 535 vehicle records (9.93%) contain missing information.

Importantly, zero previous claims are already explicitly represented in the dataset.

Therefore, a missing `previous_claims` value should not be interpreted as zero:

- `0` means that the vehicle is explicitly recorded as having no previous claims.
- A missing value means that the historical claim count is unavailable or unknown.

Replacing missing values with zero would therefore merge two different business states and introduce potentially incorrect information.

Missing `previous_claims` values are preserved as null values in the cleaned dataset.

If this variable is used in machine learning, missing-value treatment and potentially a missingness indicator can be handled later within the training preprocessing pipeline.

### Vehicle Year Missing Value Assessment

The `year` variable represents the vehicle model year and contains a relatively small proportion of missing observations.

Before deciding whether these values should be reconstructed or preserved as missing, the distribution of available vehicle years is examined across vehicle brands and models.

No statistical imputation is performed at this stage.

In [190]:
vehicles["year"].describe()

count   5,117.00
mean    2,019.09
std         3.80
min     2,010.00
25%     2,018.00
50%     2,020.00
75%     2,022.00
max     2,024.00
Name: year, dtype: float64

In [191]:
vehicle_year_summary = (
    vehicles["year"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("year")
    .reset_index(name="vehicle_count")
)

vehicle_year_summary

,year,vehicle_count
0,"2,010.00",158
1,"2,011.00",155
2,"2,012.00",155
3,"2,013.00",151
4,"2,014.00",161
5,"2,015.00",152
6,"2,016.00",141
7,"2,017.00",150
8,"2,018.00",710
9,"2,019.00",541


In [192]:
missing_year_by_vehicle = (
    vehicles.loc[
        vehicles["year"].isna()
    ]
    .groupby(
        ["brand", "model"]
    )
    .size()
    .rename("missing_year_count")
    .sort_values(ascending=False)
    .reset_index()
)

missing_year_by_vehicle

,brand,model,missing_year_count
0,Peugeot,208,34
1,Volkswagen,Polo,34
2,Renault,Megane,30
3,Renault,Captur,29
4,Peugeot,308,27
5,Volkswagen,Tiguan,24
6,Peugeot,3008,23
7,Renault,Clio,22
8,Volkswagen,Golf,19
9,BMW,X1,9


### Vehicle Year Distribution by Brand and Model

Missing vehicle years are distributed across multiple brands and models.

Before considering deterministic reconstruction, the available year distributions within each brand-model combination are examined.

If a brand-model combination appears across multiple model years, the missing year cannot be inferred reliably from the vehicle model alone and should remain missing at the general data-cleaning stage.

In [193]:
vehicle_year_by_model = (
    vehicles
    .dropna(subset=["year"])
    .groupby(["brand", "model"])["year"]
    .agg(
        unique_years="nunique",
        min_year="min",
        max_year="max",
        median_year="median",
    )
    .reset_index()
)

vehicle_year_by_model

,brand,model,unique_years,min_year,max_year,median_year
0,BMW,Serie1,15,"2,010.00","2,024.00","2,020.00"
1,BMW,Serie3,14,"2,010.00","2,024.00","2,020.50"
2,BMW,X1,15,"2,010.00","2,024.00","2,020.00"
3,Mercedes,ClasseA,14,"2,010.00","2,024.00","2,020.00"
4,Mercedes,ClasseC,14,"2,011.00","2,024.00","2,020.00"
5,Mercedes,GLA,14,"2,010.00","2,024.00","2,020.00"
6,Peugeot,208,15,"2,010.00","2,024.00","2,020.00"
7,Peugeot,3008,15,"2,010.00","2,024.00","2,020.00"
8,Peugeot,308,15,"2,010.00","2,024.00","2,020.00"
9,Renault,Captur,15,"2,010.00","2,024.00","2,020.00"


In [194]:
vehicle_year_by_model["unique_years"].value_counts().sort_index()

unique_years
14     4
15    11
Name: count, dtype: int64

In [195]:
missing_year_model_profile = (
    missing_year_by_vehicle
    .merge(
        vehicle_year_by_model,
        on=["brand", "model"],
        how="left",
        validate="one_to_one",
    )
)

missing_year_model_profile

,brand,model,missing_year_count,unique_years,min_year,max_year,median_year
0,Peugeot,208,34,15,"2,010.00","2,024.00","2,020.00"
1,Volkswagen,Polo,34,15,"2,010.00","2,024.00","2,020.00"
2,Renault,Megane,30,15,"2,010.00","2,024.00","2,020.00"
3,Renault,Captur,29,15,"2,010.00","2,024.00","2,020.00"
4,Peugeot,308,27,15,"2,010.00","2,024.00","2,020.00"
5,Volkswagen,Tiguan,24,15,"2,010.00","2,024.00","2,019.00"
6,Peugeot,3008,23,15,"2,010.00","2,024.00","2,020.00"
7,Renault,Clio,22,15,"2,010.00","2,024.00","2,020.00"
8,Volkswagen,Golf,19,15,"2,010.00","2,024.00","2,019.00"
9,BMW,X1,9,15,"2,010.00","2,024.00","2,020.00"


### Vehicle Year Missing Value Findings

Vehicle model-year information is missing for 273 records, representing 5.06% of the vehicle dataset.

The available data shows that vehicles within the same brand-model combination generally span 14 to 15 different model years, typically covering the period from 2010 to 2024.

Therefore, vehicle year cannot be reconstructed deterministically from brand and model information.

Although median or model-level imputation could be applied statistically, doing so during the general data-cleaning stage would create artificial vehicle attributes and could introduce information leakage into future machine-learning workflows.

Missing `year` values are therefore preserved as null values.

If vehicle year is required for predictive modeling, statistical imputation will be performed later within the training preprocessing pipeline.

### Vehicle Color Missing Value Assessment

The `color` variable contains missing values for a portion of the vehicle records.

Since vehicle color is a categorical attribute and cannot be reliably inferred from other vehicle characteristics, missing values are expected to remain null unless a deterministic relationship is identified.

The distribution of missing color information across vehicle brands and models is examined before making the final decision.

In [196]:
color_summary = (
    vehicles["color"]
    .value_counts(dropna=False)
    .rename_axis("color")
    .reset_index(name="vehicle_count")
)

color_summary["vehicle_pct"] = (
    color_summary["vehicle_count"]
    / len(vehicles)
    * 100
)

color_summary

,color,vehicle_count,vehicle_pct
0,White,948,17.59
1,Blue,939,17.42
2,Gray,895,16.60
3,Black,887,16.46
4,NaN,861,15.97
5,Red,860,15.96


In [197]:
missing_color_by_brand = (
    vehicles
    .assign(color_missing=vehicles["color"].isna())
    .groupby("brand")
    .agg(
        vehicle_count=("contract_id", "count"),
        missing_color=("color_missing", "sum"),
    )
)

missing_color_by_brand["missing_pct"] = (
    missing_color_by_brand["missing_color"]
    / missing_color_by_brand["vehicle_count"]
    * 100
)

missing_color_by_brand

,vehicle_count,missing_color,missing_pct
brand,,,
BMW,327,46,14.07
Mercedes,279,38,13.62
Peugeot,1594,254,15.93
Renault,1592,263,16.52
Volkswagen,1598,260,16.27


In [198]:
missing_color_by_model = (
    vehicles
    .assign(color_missing=vehicles["color"].isna())
    .groupby(["brand", "model"])
    .agg(
        vehicle_count=("contract_id", "count"),
        missing_color=("color_missing", "sum"),
    )
)

missing_color_by_model["missing_pct"] = (
    missing_color_by_model["missing_color"]
    / missing_color_by_model["vehicle_count"]
    * 100
)

missing_color_by_model.sort_values(
    "missing_pct",
    ascending=False,
)

vehicle_count  missing_color  missing_pct
brand      model                                             
Volkswagen Polo               536             94        17.54
BMW        Serie3              99             17        17.17
Volkswagen Tiguan             564             95        16.84
Peugeot    208                543             91        16.76
Renault    Clio               538             90        16.73
           Captur             509             85        16.70
Peugeot    308                539             89        16.51
Renault    Megane             545             88        16.15
Mercedes   GLA                 91             14        15.38
Peugeot    3008               512             74        14.45
BMW        X1                 112             16        14.29
Volkswagen Golf               498             71        14.26
Mercedes   ClasseC             79             11        13.92
           ClasseA            109             13        11.93
BMW        Serie1             116             13        11.21

### Vehicle Color Missing Value Findings

Missing vehicle color information is distributed relatively evenly across brands and models.

Brand-level missing rates range approximately from 14% to 17%, while model-level missing rates show a similarly broad distribution.

No brand or model exhibits a deterministic relationship that would allow missing vehicle colors to be reconstructed reliably.

Since vehicle color is an observed categorical attribute rather than a value that can be inferred from other characteristics, missing values are preserved as null.

Mode imputation is intentionally avoided because assigning the most common color would create artificial vehicle information.

If the variable is later used in machine learning, missing values may be handled as a dedicated category within the training preprocessing pipeline.

## 8. Final Clean Schema Preparation

After completing the major cleaning and standardization steps, the working datasets contain both original source fields and temporary helper variables used during validation.

The final cleaned datasets should retain:

- Business-relevant source attributes
- Standardized numeric and categorical variables
- Cleaned date fields
- Useful audit fields where appropriate

Temporary parsing and validation columns will not be included in the final analytical schema.

Separate final DataFrames are created before exporting the cleaned datasets.

In [199]:
contracts_clean = contracts[
    [
        "contract_id",
        "client_id",
        "client_name",
        "product",
        "start_date_parsed",
        "end_date_parsed",
        "annual_premium",
        "status",
        "city",
        "postal_code",
        "risk_zone",
        "client_age",
        "channel",
        "csp",
        "gender",
    ]
].copy()

In [200]:
contracts_clean = contracts_clean.rename(
    columns={
        "start_date_parsed": "start_date",
        "end_date_parsed": "end_date",
    }
)

In [201]:
contracts_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   contract_id     15000 non-null  str           
 1   client_id       15000 non-null  str           
 2   client_name     15000 non-null  str           
 3   product         15000 non-null  str           
 4   start_date      14996 non-null  datetime64[us]
 5   end_date        14995 non-null  datetime64[us]
 6   annual_premium  15000 non-null  Float64       
 7   status          15000 non-null  str           
 8   city            15000 non-null  string        
 9   postal_code     15000 non-null  string        
 10  risk_zone       15000 non-null  str           
 11  client_age      13735 non-null  float64       
 12  channel         15000 non-null  str           
 13  csp             13228 non-null  str           
 14  gender          11904 non-null  str           
dtypes: Float64(1)

In [202]:
contracts_clean.head()

,contract_id,client_id,client_name,product,start_date,end_date,annual_premium,status,city,postal_code,risk_zone,client_age,channel,csp,gender
0,CTR_000001,CLI_000001,Pascal Dubois,Life,2023-08-11,2024-09-08,"1,974.98",Renewed,Paris,75001,High,50.00,Agency,NaN,Female
1,CTR_000002,CLI_000002,Sophie Simon,Auto,2025-08-12,2026-08-15,620.93,Active,Bordeaux,33000,Medium,43.00,Phone,Worker,Female
2,CTR_000003,CLI_000003,Olivier Durand,Auto,2025-06-14,2026-06-30,"1,568.11",Suspended,Paris,75001,High,63.00,Broker,NaN,Male
3,CTR_000004,CLI_000004,Sandrine Michel,Life,2023-04-17,2024-04-20,"1,752.17",Renewed,Bordeaux,33000,Medium,54.00,Broker,Manager,Male
4,CTR_000005,CLI_000005,Pierre Durand,Auto,2025-03-02,2026-02-26,977.59,Suspended,Marseille,13000,Medium,39.00,Web,Employee,Male


In [203]:
print(
    "Raw rows:",
    len(contracts_raw)
)

print(
    "Clean rows:",
    len(contracts_clean)
)

print(
    "Clean columns:",
    contracts_clean.shape[1]
)

print(
    "Duplicate contract IDs:",
    contracts_clean["contract_id"].duplicated().sum()
)

Raw rows: 15000
Clean rows: 15000
Clean columns: 15
Duplicate contract IDs: 0


In [204]:
contracts_clean = contracts_clean.rename(
    columns={
        "status": "contract_status",
    }
)

contracts_clean.head()

,contract_id,client_id,client_name,product,start_date,end_date,annual_premium,contract_status,city,postal_code,risk_zone,client_age,channel,csp,gender
0,CTR_000001,CLI_000001,Pascal Dubois,Life,2023-08-11,2024-09-08,"1,974.98",Renewed,Paris,75001,High,50.00,Agency,NaN,Female
1,CTR_000002,CLI_000002,Sophie Simon,Auto,2025-08-12,2026-08-15,620.93,Active,Bordeaux,33000,Medium,43.00,Phone,Worker,Female
2,CTR_000003,CLI_000003,Olivier Durand,Auto,2025-06-14,2026-06-30,"1,568.11",Suspended,Paris,75001,High,63.00,Broker,NaN,Male
3,CTR_000004,CLI_000004,Sandrine Michel,Life,2023-04-17,2024-04-20,"1,752.17",Renewed,Bordeaux,33000,Medium,54.00,Broker,Manager,Male
4,CTR_000005,CLI_000005,Pierre Durand,Auto,2025-03-02,2026-02-26,977.59,Suspended,Marseille,13000,Medium,39.00,Web,Employee,Male


### Final Contracts Schema Findings

The final contracts dataset contains all 15,000 original contract records without row loss.

Key cleaning results include:

- Contract dates are stored as standardized datetime values.
- Annual premiums are stored as numeric values.
- `city_postal` has been separated into `city` and `postal_code`.
- Gender categories have been standardized.
- Missing customer attributes remain null where they cannot be reconstructed reliably.
- Ambiguous contract dates that could not be resolved deterministically remain missing.
- `contract_id` remains unique and can serve as the primary key of the contracts table.

The cleaned contracts dataset is now suitable for relational database storage, analytical reporting, and downstream feature engineering.

### Final Claims Schema

The final claims dataset retains the cleaned business attributes together with selected audit and analytical fields.

Raw date strings and temporary parsing columns are excluded from the final schema.

The following derived fields are retained:

- `declaration_lag_days`: number of days between claim occurrence and declaration
- `claim_dates_swapped`: audit flag indicating records where occurrence and declaration dates were corrected by reversing their original roles

The audit flag provides transparency for the claim-date corrections applied during data cleaning.

In [205]:
claims_clean = claims[
    [
        "claim_id",
        "contract_id",
        "occurrence_date_parsed",
        "declaration_date_parsed",
        "claim_type",
        "damage_amount",
        "indemnified_amount",
        "status",
        "expert_id",
        "liability",
        "declaration_lag_days",
        "claim_dates_swapped",
    ]
].copy()

In [206]:
claims_clean = claims_clean.rename(
    columns={
        "occurrence_date_parsed": "occurrence_date",
        "declaration_date_parsed": "declaration_date",
        "status": "claim_status",
    }
)

In [207]:
claims_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   claim_id              155 non-null    str           
 1   contract_id           155 non-null    str           
 2   occurrence_date       155 non-null    datetime64[us]
 3   declaration_date      154 non-null    datetime64[us]
 4   claim_type            155 non-null    str           
 5   damage_amount         155 non-null    Float64       
 6   indemnified_amount    90 non-null     Float64       
 7   claim_status          155 non-null    str           
 8   expert_id             117 non-null    str           
 9   liability             108 non-null    str           
 10  declaration_lag_days  154 non-null    float64       
 11  claim_dates_swapped   155 non-null    bool          
dtypes: Float64(2), bool(1), datetime64[us](2), float64(1), str(6)
memory usage: 13.9 KB


In [208]:
claims_clean.head()

,claim_id,contract_id,occurrence_date,declaration_date,claim_type,damage_amount,indemnified_amount,claim_status,expert_id,liability,declaration_lag_days,claim_dates_swapped
0,CLM_0000001,CTR_008899,2023-10-02,2023-11-26,Theft,"15,213.03","10,977.27",Closed,EXP_013,Third_party,55.00,True
1,CLM_0000002,CTR_001770,2025-08-26,2025-08-28,Fire,"2,321.55",<NA>,Expert_review,EXP_013,Third_party,2.00,False
2,CLM_0000003,CTR_002653,2024-09-26,2024-10-25,Fire,"1,762.45","1,030.68",Closed,EXP_001,Third_party,29.00,True
3,CLM_0000004,CTR_002271,2024-04-16,2024-05-27,Collision,"3,144.06","2,119.97",Closed,NaN,Insured,41.00,False
4,CLM_0000005,CTR_000649,2025-03-08,2025-04-12,Collision,"3,715.18",<NA>,Expert_review,EXP_007,Third_party,35.00,False


In [209]:
print(
    "Raw rows:",
    len(claims_raw)
)

print(
    "Clean rows:",
    len(claims_clean)
)

print(
    "Duplicate claim IDs:",
    claims_clean["claim_id"].duplicated().sum()
)

print(
    "Missing occurrence dates:",
    claims_clean["occurrence_date"].isna().sum()
)

print(
    "Missing declaration dates:",
    claims_clean["declaration_date"].isna().sum()
)

print(
    "Negative declaration lags:",
    (claims_clean["declaration_lag_days"] < 0).sum()
)

print(
    "Date reversal flags:",
    claims_clean["claim_dates_swapped"].sum()
)

Raw rows: 155
Clean rows: 155
Duplicate claim IDs: 0
Missing occurrence dates: 0
Missing declaration dates: 1
Negative declaration lags: 0
Date reversal flags: 64


### Final Claims Schema Findings

The final claims dataset retains all 155 original claim records without row loss.

Key cleaning results include:

- Claim occurrence dates are fully standardized.
- 154 of 155 declaration dates were resolved.
- Chronologically inconsistent claim dates were corrected using the validated date-reversal rule.
- `claim_dates_swapped` is retained as an audit field to document corrected records.
- `declaration_lag_days` provides an analytical measure of the delay between claim occurrence and declaration.
- Monetary claim variables are stored as numeric values.
- Structurally missing indemnification amounts remain null.
- Missing `expert_id` and `liability` values are preserved where they cannot be reconstructed reliably.
- `claim_id` remains unique and can serve as the primary key of the claims table.

The cleaned claims dataset is now suitable for relational database storage, claims reporting, and downstream machine-learning feature engineering.

### Final Vehicles Schema

The final vehicles dataset retains standardized vehicle attributes while excluding temporary parsing variables.

The raw `power` field is replaced by:

- `power_hp`: standardized vehicle power expressed in horsepower
- `power_unit`: the original detected unit category used for auditability

Temporary parsing fields such as `power_value_raw` are excluded from the final schema.

Missing vehicle attributes remain null where they cannot be reconstructed deterministically.

In [210]:
vehicles_clean = vehicles[
    [
        "contract_id",
        "brand",
        "model",
        "year",
        "power_hp",
        "power_unit",
        "fuel_type",
        "current_value",
        "color",
        "usage",
        "previous_claims",
    ]
].copy()

In [211]:
vehicles_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 5390 entries, 0 to 5389
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   contract_id      5390 non-null   str    
 1   brand            5390 non-null   str    
 2   model            5390 non-null   str    
 3   year             5117 non-null   float64
 4   power_hp         4490 non-null   Float64
 5   power_unit       4490 non-null   string 
 6   fuel_type        5390 non-null   str    
 7   current_value    5390 non-null   Float64
 8   color            4529 non-null   str    
 9   usage            5390 non-null   str    
 10  previous_claims  4855 non-null   float64
dtypes: Float64(2), float64(2), str(6), string(1)
memory usage: 473.9 KB


In [212]:
vehicles_clean.head()

,contract_id,brand,model,year,power_hp,power_unit,fuel_type,current_value,color,usage,previous_claims
0,CTR_000003,BMW,Serie1,"2,022.00",128.00,HP,Gasoline,"29,567.77",Gray,Mixed,0.00
1,CTR_000005,Renault,Megane,"2,024.00",150.00,HP,Hybrid,"14,873.92",Black,Personal,0.00
2,CTR_000012,Peugeot,208,"2,020.00",175.00,Unitless,Hybrid,"8,362.30",White,Professional,2.00
3,CTR_000015,Renault,Captur,"2,024.00",<NA>,<NA>,Electric,"14,849.30",NaN,Mixed,1.00
4,CTR_000016,Renault,Megane,NaN,176.00,HP,Gasoline,"14,522.08",Blue,Personal,1.00


In [213]:
print(
    "Raw rows:",
    len(vehicles_raw)
)

print(
    "Clean rows:",
    len(vehicles_clean)
)

print(
    "Duplicate contract IDs:",
    vehicles_clean["contract_id"].duplicated().sum()
)

print(
    "Missing year:",
    vehicles_clean["year"].isna().sum()
)

print(
    "Missing power_hp:",
    vehicles_clean["power_hp"].isna().sum()
)

print(
    "Missing color:",
    vehicles_clean["color"].isna().sum()
)

print(
    "Missing previous_claims:",
    vehicles_clean["previous_claims"].isna().sum()
)

Raw rows: 5390
Clean rows: 5390
Duplicate contract IDs: 0
Missing year: 273
Missing power_hp: 900
Missing color: 861
Missing previous_claims: 535


### Final Vehicles Schema Findings

The final vehicles dataset contains all 5,390 original vehicle records without row loss.

Key cleaning results include:

- Vehicle monetary values are stored as numeric values.
- Vehicle power has been standardized to horsepower through `power_hp`.
- The original detected power representation is preserved in `power_unit` for auditability.
- Missing vehicle year, power, color, and previous-claim information remains null where it cannot be reconstructed reliably.
- Explicit zero values in `previous_claims` remain distinct from missing values.
- `contract_id` remains unique within the vehicles dataset.

The cleaned vehicles dataset is now suitable for relational database storage, Auto insurance analytics, and future machine-learning feature engineering.

## 9. Final Data Type Standardization

Before exporting the cleaned datasets, selected variables are converted to data types that better reflect their business meaning.

Integer-like variables containing missing values are converted to Pandas nullable `Int64` rather than ordinary integers.

This allows integer semantics to be preserved while still supporting null values.

Monetary and continuous variables remain floating-point values, date fields remain datetime variables, and identifiers remain string-based.

In [214]:
integer_candidates = {
    "contracts.client_age": contracts_clean["client_age"],
    "claims.declaration_lag_days": claims_clean["declaration_lag_days"],
    "vehicles.year": vehicles_clean["year"],
    "vehicles.previous_claims": vehicles_clean["previous_claims"],
}

integer_validation = []

for name, series in integer_candidates.items():

    non_missing = series.dropna()

    integer_validation.append(
        {
            "column": name,
            "non_missing_count": len(non_missing),
            "non_integer_values": (
                non_missing.mod(1).ne(0)
            ).sum(),
        }
    )

integer_validation = pd.DataFrame(integer_validation)

integer_validation

,column,non_missing_count,non_integer_values
0,contracts.client_age,13735,0
1,claims.declaration_lag_days,154,0
2,vehicles.year,5117,0
3,vehicles.previous_claims,4855,0


In [215]:
contracts_clean["client_age"] = (
    contracts_clean["client_age"]
    .astype("Int64")
)

claims_clean["declaration_lag_days"] = (
    claims_clean["declaration_lag_days"]
    .astype("Int64")
)

vehicles_clean["year"] = (
    vehicles_clean["year"]
    .astype("Int64")
)

vehicles_clean["previous_claims"] = (
    vehicles_clean["previous_claims"]
    .astype("Int64")
)

In [216]:
contracts_clean["postal_code"] = (
    contracts_clean["postal_code"]
    .astype("string")
)

In [217]:
print("CONTRACTS")
print(
    contracts_clean[
        [
            "client_age",
            "annual_premium",
            "postal_code",
            "start_date",
            "end_date",
        ]
    ].dtypes
)

print("\nCLAIMS")
print(
    claims_clean[
        [
            "damage_amount",
            "indemnified_amount",
            "declaration_lag_days",
            "occurrence_date",
            "declaration_date",
            "claim_dates_swapped",
        ]
    ].dtypes
)

print("\nVEHICLES")
print(
    vehicles_clean[
        [
            "year",
            "power_hp",
            "current_value",
            "previous_claims",
        ]
    ].dtypes
)

CONTRACTS
client_age                 Int64
annual_premium           Float64
postal_code               string
start_date        datetime64[us]
end_date          datetime64[us]
dtype: object

CLAIMS
damage_amount                  Float64
indemnified_amount             Float64
declaration_lag_days             Int64
occurrence_date         datetime64[us]
declaration_date        datetime64[us]
claim_dates_swapped               bool
dtype: object

VEHICLES
year                 Int64
power_hp           Float64
current_value      Float64
previous_claims      Int64
dtype: object


### Final Data Type Findings

The cleaned datasets were successfully standardized using data types that reflect the business meaning of each variable.

Key changes include:

- Customer age, vehicle year, previous claim count, and declaration lag were converted to nullable integer types.
- Monetary and continuous variables remain nullable floating-point values.
- Postal codes are preserved as strings because they represent identifiers rather than numeric measurements.
- Date variables are stored as datetime values.
- The claim date-correction audit field remains boolean.

These data types provide a cleaner foundation for MySQL schema design, Power BI integration, and downstream machine-learning workflows.

## 10. Final Data Quality Validation

A final set of validation checks is performed before the cleaned datasets are exported.

The validation focuses on:

- Row preservation
- Primary key uniqueness
- Referential integrity
- Date consistency
- Monetary value validity
- Customer and vehicle attribute ranges
- Claim chronology
- Product-specific vehicle relationships

The objective is to ensure that the cleaning process has not introduced structural or logical inconsistencies.

In [218]:
validation_results = {}

In [219]:
validation_results["contracts_row_count_preserved"] = (
    len(contracts_clean) == len(contracts_raw)
)

validation_results["claims_row_count_preserved"] = (
    len(claims_clean) == len(claims_raw)
)

validation_results["vehicles_row_count_preserved"] = (
    len(vehicles_clean) == len(vehicles_raw)
)

In [220]:
validation_results["contract_id_unique"] = (
    contracts_clean["contract_id"].is_unique
)

validation_results["claim_id_unique"] = (
    claims_clean["claim_id"].is_unique
)

validation_results["vehicle_contract_id_unique"] = (
    vehicles_clean["contract_id"].is_unique
)

In [221]:
validation_results["contract_id_no_missing"] = (
    contracts_clean["contract_id"].notna().all()
)

validation_results["claim_id_no_missing"] = (
    claims_clean["claim_id"].notna().all()
)

validation_results["vehicle_contract_id_no_missing"] = (
    vehicles_clean["contract_id"].notna().all()
)

In [222]:
contract_ids = set(
    contracts_clean["contract_id"]
)

validation_results["claims_fk_valid"] = (
    claims_clean["contract_id"]
    .isin(contract_ids)
    .all()
)

validation_results["vehicles_fk_valid"] = (
    vehicles_clean["contract_id"]
    .isin(contract_ids)
    .all()
)

In [223]:
valid_contract_date_rows = (
    contracts_clean["start_date"].notna()
    & contracts_clean["end_date"].notna()
)

validation_results["contract_dates_chronological"] = (
    (
        contracts_clean.loc[
            valid_contract_date_rows,
            "end_date"
        ]
        >=
        contracts_clean.loc[
            valid_contract_date_rows,
            "start_date"
        ]
    ).all()
)

In [224]:
valid_claim_date_rows = (
    claims_clean["occurrence_date"].notna()
    & claims_clean["declaration_date"].notna()
)

validation_results["claim_dates_chronological"] = (
    (
        claims_clean.loc[
            valid_claim_date_rows,
            "declaration_date"
        ]
        >=
        claims_clean.loc[
            valid_claim_date_rows,
            "occurrence_date"
        ]
    ).all()
)

In [225]:
validation_results["claim_lag_valid"] = (
    claims_clean["declaration_lag_days"]
    .dropna()
    .between(0, 62)
    .all()
)

In [226]:
validation_results["annual_premium_non_negative"] = (
    contracts_clean["annual_premium"]
    .dropna()
    .ge(0)
    .all()
)

validation_results["damage_amount_non_negative"] = (
    claims_clean["damage_amount"]
    .dropna()
    .ge(0)
    .all()
)

validation_results["indemnified_amount_non_negative"] = (
    claims_clean["indemnified_amount"]
    .dropna()
    .ge(0)
    .all()
)

validation_results["current_value_non_negative"] = (
    vehicles_clean["current_value"]
    .dropna()
    .ge(0)
    .all()
)

In [227]:
validation_results["client_age_valid"] = (
    contracts_clean["client_age"]
    .dropna()
    .between(18, 100)
    .all()
)

validation_results["vehicle_year_valid"] = (
    vehicles_clean["year"]
    .dropna()
    .between(1950, 2026)
    .all()
)

validation_results["previous_claims_non_negative"] = (
    vehicles_clean["previous_claims"]
    .dropna()
    .ge(0)
    .all()
)

validation_results["power_hp_positive"] = (
    vehicles_clean["power_hp"]
    .dropna()
    .gt(0)
    .all()
)

In [228]:
final_validation = (
    pd.Series(
        validation_results,
        name="passed"
    )
    .rename_axis("validation_check")
    .reset_index()
)

final_validation

,validation_check,passed
0,contracts_row_count_preserved,True
1,claims_row_count_preserved,True
2,vehicles_row_count_preserved,True
3,contract_id_unique,True
4,claim_id_unique,True
5,vehicle_contract_id_unique,True
6,contract_id_no_missing,True
7,claim_id_no_missing,True
8,vehicle_contract_id_no_missing,True
9,claims_fk_valid,True


In [229]:
failed_checks = final_validation.loc[
    ~final_validation["passed"]
]

print(
    f"Passed checks: "
    f"{final_validation['passed'].sum()} / "
    f"{len(final_validation)}"
)

display(failed_checks)

Passed checks: 22 / 22


,validation_check,passed


### Final Validation Findings

All 22 final data-quality validation checks passed successfully.

The validation confirms that:

- All original rows were preserved.
- Primary keys remain unique and non-null.
- Foreign-key relationships remain valid.
- Contract and claim dates satisfy the defined chronological rules where dates are available.
- Claim declaration delays remain within the validated range.
- Monetary values are non-negative.
- Customer and vehicle attributes remain within valid ranges.
- No invalid vehicle power or previous-claim values were introduced.

The cleaned datasets are therefore considered ready for export and downstream database integration.

## 11. Export Cleaned Datasets

The validated cleaned datasets are exported to the `data/processed/` directory.

These files represent the standardized analytical layer of the project and will be used for:

- MySQL database loading
- SQL analysis
- Power BI reporting
- Feature engineering
- Machine learning workflows

Raw source files remain unchanged under `data/raw/`.

In [230]:
CONTRACTS_CLEAN_PATH = (
    PROCESSED_DATA_DIR / "contracts_clean.csv"
)

CLAIMS_CLEAN_PATH = (
    PROCESSED_DATA_DIR / "claims_clean.csv"
)

VEHICLES_CLEAN_PATH = (
    PROCESSED_DATA_DIR / "vehicles_clean.csv"
)

In [231]:
contracts_clean.to_csv(
    CONTRACTS_CLEAN_PATH,
    index=False,
    date_format="%Y-%m-%d",
)

claims_clean.to_csv(
    CLAIMS_CLEAN_PATH,
    index=False,
    date_format="%Y-%m-%d",
)

vehicles_clean.to_csv(
    VEHICLES_CLEAN_PATH,
    index=False,
)

In [232]:
export_files = {
    "contracts": CONTRACTS_CLEAN_PATH,
    "claims": CLAIMS_CLEAN_PATH,
    "vehicles": VEHICLES_CLEAN_PATH,
}

for name, path in export_files.items():
    print(
        f"{name:<10} "
        f"exists={path.exists()} | "
        f"{path.name}"
    )

contracts  exists=True | contracts_clean.csv
claims     exists=True | claims_clean.csv
vehicles   exists=True | vehicles_clean.csv


In [233]:
contracts_export_check = pd.read_csv(
    CONTRACTS_CLEAN_PATH
)

claims_export_check = pd.read_csv(
    CLAIMS_CLEAN_PATH
)

vehicles_export_check = pd.read_csv(
    VEHICLES_CLEAN_PATH
)

export_validation = pd.DataFrame(
    {
        "dataset": [
            "contracts",
            "claims",
            "vehicles",
        ],
        "memory_rows": [
            len(contracts_clean),
            len(claims_clean),
            len(vehicles_clean),
        ],
        "exported_rows": [
            len(contracts_export_check),
            len(claims_export_check),
            len(vehicles_export_check),
        ],
    }
)

export_validation["rows_match"] = (
    export_validation["memory_rows"]
    == export_validation["exported_rows"]
)

export_validation

,dataset,memory_rows,exported_rows,rows_match
0,contracts,15000,15000,True
1,claims,155,155,True
2,vehicles,5390,5390,True


In [234]:
assert export_validation["rows_match"].all()

print("All cleaned datasets exported successfully.")

All cleaned datasets exported successfully.


## 12. Data Cleaning Summary

The data-cleaning stage of the Insurance Analytics Platform has been completed successfully.

The major transformations performed in this notebook include:

- Standardization of monetary variables
- Resolution and validation of mixed-format contract dates
- Correction of claim occurrence/declaration date reversals
- Preservation of genuinely ambiguous dates
- Standardization of gender categories
- Separation of city and postal-code information
- Standardization of vehicle power into horsepower
- Business-aware treatment of missing values
- Final data-type standardization
- Primary and foreign key validation
- Business-rule validation

All 22 final data-quality checks passed successfully.

The cleaned datasets were exported to:

- `data/processed/contracts_clean.csv`
- `data/processed/claims_clean.csv`
- `data/processed/vehicles_clean.csv`

These datasets now form the cleaned analytical layer of the project and are ready to be loaded into MySQL.

The next stage of the project will focus on relational database design, table creation, and loading the cleaned datasets into MySQL.

In [2]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROJECT_ROOT = PROJECT_ROOT.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CURRENT_DIR :", CURRENT_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("sys.path[0] :", sys.path[0])

print(
    "src exists:",
    (PROJECT_ROOT / "src").exists()
)

print(
    "src/__init__.py exists:",
    (PROJECT_ROOT / "src" / "__init__.py").exists()
)

print(
    "cleaning.py exists:",
    (
        PROJECT_ROOT
        / "src"
        / "preprocessing"
        / "cleaning.py"
    ).exists()
)

CURRENT_DIR : c:\Users\okand\Desktop\Projects\insurance-analytics-platform\notebooks
PROJECT_ROOT: C:\Users\okand\Desktop\Projects\insurance-analytics-platform
sys.path[0] : C:\Users\okand\Desktop\Projects\insurance-analytics-platform
src exists: True
src/__init__.py exists: True
cleaning.py exists: True


In [247]:
from src.preprocessing.cleaning import (
    get_ambiguous_date_candidates,
    parse_monetary_series,
    parse_unambiguous_date,
    split_city_postal,
    standardize_gender
)

In [249]:
import importlib
import src.preprocessing.cleaning as cleaning

importlib.reload(cleaning)

<module 'src.preprocessing.cleaning' from 'C:\\Users\\okand\\Desktop\\Projects\\insurance-analytics-platform\\src\\preprocessing\\cleaning.py'>

In [250]:
contracts_clean_module = cleaning.clean_contracts(
    contracts_raw
)

In [252]:
print(
    "Shape:",
    contracts_clean_module.shape
)

print(
    "Duplicate contract IDs:",
    contracts_clean_module["contract_id"]
    .duplicated()
    .sum()
)

print(
    "Missing start dates:",
    contracts_clean_module["start_date"]
    .isna()
    .sum()
)

print(
    "Missing end dates:",
    contracts_clean_module["end_date"]
    .isna()
    .sum()
)

Shape: (15000, 15)
Duplicate contract IDs: 0
Missing start dates: 4
Missing end dates: 5


In [253]:
print(
    "Same result as notebook:",
    contracts_clean_module.equals(
        contracts_clean
    )
)

Same result as notebook: True


In [254]:
contracts_clean_module.compare(
    contracts_clean
)

Empty DataFrame
Columns: []
Index: []

In [255]:
print(
    "Same result as notebook:",
    contracts_clean_module.equals(
        contracts_clean
    )
)

Same result as notebook: True


In [260]:
import importlib
import src.preprocessing.cleaning as cleaning

importlib.reload(cleaning)

<module 'src.preprocessing.cleaning' from 'C:\\Users\\okand\\Desktop\\Projects\\insurance-analytics-platform\\src\\preprocessing\\cleaning.py'>

In [261]:
claims_clean_module = cleaning.clean_claims(
    claims_raw
)

In [262]:
print("Shape:", claims_clean_module.shape)

print(
    "Missing occurrence dates:",
    claims_clean_module["occurrence_date"].isna().sum()
)

print(
    "Missing declaration dates:",
    claims_clean_module["declaration_date"].isna().sum()
)

print(
    "Date reversal flags:",
    claims_clean_module["claim_dates_swapped"].sum()
)

print(
    "Negative declaration lags:",
    (
        claims_clean_module["declaration_lag_days"] < 0
    ).sum()
)

Shape: (155, 12)
Missing occurrence dates: 0
Missing declaration dates: 1
Date reversal flags: 64
Negative declaration lags: 0


In [263]:
print(
    "Same result as notebook:",
    claims_clean_module.equals(
        claims_clean
    )
)

Same result as notebook: True


In [264]:
import importlib
import src.preprocessing.cleaning as cleaning

importlib.reload(cleaning)

<module 'src.preprocessing.cleaning' from 'C:\\Users\\okand\\Desktop\\Projects\\insurance-analytics-platform\\src\\preprocessing\\cleaning.py'>

In [265]:
vehicles_clean_module = cleaning.clean_vehicles(
    vehicles_raw
)

In [266]:
print(
    "Shape:",
    vehicles_clean_module.shape
)

print(
    "Duplicate contract IDs:",
    vehicles_clean_module["contract_id"]
    .duplicated()
    .sum()
)

print(
    "Missing year:",
    vehicles_clean_module["year"]
    .isna()
    .sum()
)

print(
    "Missing power_hp:",
    vehicles_clean_module["power_hp"]
    .isna()
    .sum()
)

print(
    "Missing color:",
    vehicles_clean_module["color"]
    .isna()
    .sum()
)

print(
    "Missing previous_claims:",
    vehicles_clean_module["previous_claims"]
    .isna()
    .sum()
)

Shape: (5390, 11)
Duplicate contract IDs: 0
Missing year: 273
Missing power_hp: 900
Missing color: 861
Missing previous_claims: 535


In [267]:
print(
    "Same result as notebook:",
    vehicles_clean_module.equals(
        vehicles_clean
    )
)

Same result as notebook: True


In [268]:
import importlib
import src.preprocessing.cleaning as cleaning

importlib.reload(cleaning)

(
    contracts_clean_all,
    claims_clean_all,
    vehicles_clean_all,
) = cleaning.clean_all_datasets(
    contracts_raw,
    claims_raw,
    vehicles_raw,
)

In [269]:
print(
    "Contracts match:",
    contracts_clean_all.equals(contracts_clean)
)

print(
    "Claims match:",
    claims_clean_all.equals(claims_clean)
)

print(
    "Vehicles match:",
    vehicles_clean_all.equals(vehicles_clean)
)

Contracts match: True
Claims match: True
Vehicles match: True


In [270]:
print(
    "Raw contracts unchanged:",
    contracts_raw.equals(
        pd.read_csv(CONTRACTS_PATH)
    )
)

print(
    "Raw claims unchanged:",
    claims_raw.equals(
        pd.read_csv(CLAIMS_PATH)
    )
)

print(
    "Raw vehicles unchanged:",
    vehicles_raw.equals(
        pd.read_csv(VEHICLES_PATH)
    )
)

Raw contracts unchanged: True
Raw claims unchanged: True
Raw vehicles unchanged: True
